# Notebook 2 — AURKB classical QSAR and reproducibility workflow

**Article:** Chemotype-aware QSAR–GCN consensus screening and molecular modeling prioritize natural-product-derived AURKB-targeting candidates for pancreatic ductal adenocarcinoma

**Target journal:** Molecular Diversity

**Authors:** The author list, affiliations and corresponding-author details are not recorded in the
manuscript file supplied with this repository and are therefore not reproduced here. They are given in
the submitted manuscript.

**Purpose of this notebook:** Reproduce the descriptor-based classical QSAR workflow under provenance control: curation audit, hash verification of the frozen 217-descriptor matrix, leakage-controlled model comparison, sigmoid-calibrated random held-out validation, Y-randomization, applicability-domain reconstruction, the corrected scaffold-disjoint validation branch, and preservation of the exact QSAR-to-GCN transfer.

**Workflow stage:** Stage 2 of 3 in the computational screening chain.

**Input:** Curated AURKB ChEMBL bioactivity set (3,248 raw records; 1,854 curated compounds), the frozen 217-descriptor matrix, archived validation summaries, preserved preprocessing/calibration/model artifacts, and the 86,056-compound prescreened library from Notebook 1.

**Output:** Verification of the exact 6,232-row QSAR→GCN transfer snapshot (SHA-256 `8821947e92d8b209d291b627bae3b1d5068c6e2f1c0154f695abf0e73673f4df`), reproduction status tables, environment capture and a SHA-256 artifact inventory.

**Relationship to the other stages:** Consumes the Notebook 1 library. The 6,232-row transfer snapshot is the authoritative input of Notebook 3. Values fitted afresh in the current software environment are diagnostics of software-version sensitivity and do not replace the historical production values; the historical run identifier is `run_20260715T124827Z_79cebaa2`.

**Frozen-artifact notice:** The scientific state of this study is frozen. This file is a
presentation-only copy of the executed original notebook. Only Markdown text and Python comments were
edited; executable code, cell order, cell identifiers, execution counts and all stored outputs are
unchanged and were not re-executed for this repository. Historical manuscript-authoritative artifacts
are never replaced by values recalculated in a current software environment.

**Executed outputs:** The outputs stored in this notebook are the outputs of the original recorded
execution and are retained for provenance, including environment/version records, warnings and
software-drift evidence.

---


# AURKB Classical QSAR — Final Authoritative Reproduction Notebook

**Permanent archive architecture: Option A — one self-contained notebook with two isolated scientific branches.**

This notebook is the authoritative replacement for the unverified historical Classical QSAR notebook. It contains:

1. **Production/random-split QSAR branch** — curation audit, frozen descriptor verification, leakage-controlled model comparison, calibrated random held-out validation, Y-randomization, production checkpoint preservation, applicability-domain reconstruction, and QSAR-to-GCN transfer provenance.
2. **Corrected scaffold-disjoint validation branch** — the validation-only May 25 addendum, replayed from its train-only model and external sigmoid calibrator with zero scaffold overlap.

A single notebook is scientifically preferable here because it provides one execution graph, one output root, one manifest, and no cross-notebook hidden state. The two branches remain explicitly separated: the corrected scaffold model is **never** used to replace the production QSAR screening model.

## Integrity rules

- The verified Prescreening and GCN notebooks are outside scope and are not modified.
- Uploaded ZIP archives and all original project files are read-only.
- Every new file is written under a unique run folder inside `QSAR_Final_Reproduction/`.
- Exact historical intermediates are hash-verified before use.
- Regenerated, checkpoint-replayed, archived-only, and irreproducible results are labeled separately.
- No numerical value is altered merely to agree with the manuscript.

## Execution profiles

- `full` — retrains the random-split models, calibrated held-out model, and 30-permutation Y-randomization.
- `checkpoint_replay` — validates frozen data and archived checkpoints without expensive retraining; intended only for installation/smoke testing.

The notebook defaults to `full`. Set the environment variable `QSAR_EXECUTION_PROFILE=checkpoint_replay` only for a fast technical check.

## Cross-platform fresh-training policy

Exact assertions remain mandatory for immutable input hashes and archived checkpoint inference. Fresh fitting of XGBoost, scikit-learn calibration, random forests, and permutation tests is compared against the historical evidence but is not assumed to be bitwise identical across Windows/Linux or package builds. When fresh fitting drifts, the notebook exports the current-environment results as diagnostics, preserves the hash-verified historical manuscript layer, records versions and absolute differences, and continues without altering reported values. Set `STRICT_FRESH_RETRAINING_MATCH=True` only in a deliberately reconstructed historical environment.


> **VS Code release marker:** `AURKB-QSAR-VSCODE-DRIFT-SAFE-v2`  
> In `QSAR-07`, fresh XGBoost metrics are compared and exported; they are **not** asserted against historical values. 
> If a traceback contains `expected_exact` or `compare_numeric(actual_row[metric_name]...)`, you opened an older notebook.


## VS Code local execution

This edition is configured for the **VS Code Jupyter extension** and a normal local Python environment. It does not require Google Colab.

### Recommended project layout

Place this notebook and the required immutable archives in one project folder:

```text
AURKB_QSAR_Project/
├── 02_AURKB_Classical_QSAR_Final_Authoritative_Reproduction_VSCode_DRIFT_SAFE_v2.ipynb
├── requirements-vscode.txt
├── LOCKED_QSAR_GCN_RESULTS_DO_NOT_OVERWRITE(1).zip
├── Fixing_scafold_validation.zip
└── Manuscript and Supplimenary files.zip   # optional but recommended
```

The archives may instead be kept in an `Inputs/`, `Input/`, `Archives/`, or `Data/` subfolder. Their exact paths may be entered in the configuration cell below. This edition also searches common Windows folders recursively and can open a file-selection dialog when the archives are not found automatically.

### VS Code setup

1. Open the **project folder**, not only the notebook file, in VS Code.
2. Install the Microsoft **Python** and **Jupyter** extensions.
3. Select a writable Python kernel or virtual environment.
4. Run the configuration cell and then the dependency-bootstrap cell. If the archives are elsewhere, either paste their exact Windows paths into the configuration cell or select them in the file dialogs when prompted.
5. When required packages are missing, the bootstrap installs the validated dependency set into the **currently selected kernel**.
6. Continue running the notebook from top to bottom.

For the cleanest permanent environment, creating a dedicated `.venv` remains recommended. The bootstrap is included so a fresh VS Code kernel no longer fails immediately with `ModuleNotFoundError: No module named 'numpy'`.

Every generated scientific output remains under a unique folder inside `QSAR_Final_Reproduction/`. Existing project files and source archives are never overwritten.

In [2]:
# QSAR-VSCODE-CONFIG — User-editable local settings
NOTEBOOK_RELEASE = "AURKB-QSAR-VSCODE-DRIFT-SAFE-v2"
print("Notebook release:", NOTEBOOK_RELEASE)

#
# Recommended Windows examples:
# PROJECT_ROOT = r"C:\Users\Prottoy\Documents\AURKB_QSAR_Project"
# LOCKED_ZIP_PATH = r"C:\path\to\LOCKED_QSAR_GCN_RESULTS_DO_NOT_OVERWRITE(1).zip"
# SCAFFOLD_ZIP_PATH = r"C:\path\to\Fixing_scafold_validation.zip"
#
# Raw strings (the leading r) prevent Windows backslashes from being treated
# as escape characters. Leave a setting as None to use automatic discovery.

PROJECT_ROOT = None
INPUT_ARCHIVE_DIR = None

LOCKED_ZIP_PATH = None
SCAFFOLD_ZIP_PATH = None
MANUSCRIPT_ZIP_PATH = None

# Optional additional folders to search. Add any location where the ZIP files
# may have been saved. Both raw strings and pathlib.Path values are accepted.
ARCHIVE_SEARCH_ROOTS = [
    # r"C:\Users\Prottoy\Downloads",
    # r"D:\AURKB_Project",
]

# Search the opened project, its parents, common project subfolders, and common
# Windows/macOS/Linux user folders. ZIPs are validated by their internal file
# structure, so harmless filename changes such as "(2)" are accepted.
RECURSIVE_ARCHIVE_SEARCH = True

# When automatic discovery fails in local VS Code, open native file-selection
# dialogs. Set False for headless/remote execution without a desktop session.
ENABLE_INTERACTIVE_ARCHIVE_PICKER = True

# Dependency bootstrap for a fresh VS Code kernel.
# When True, missing scientific packages are installed into sys.executable,
# which is the interpreter backing the currently selected notebook kernel.
AUTO_INSTALL_MISSING_DEPENDENCIES = True

# Prefer the adjacent requirements-vscode.txt file when available. If it is
# absent, the notebook uses the embedded validated package specifications.
PREFER_LOCAL_REQUIREMENTS_FILE = True

# "full" reproduces all supported model training/validation.
# "checkpoint_replay" performs a faster technical validation of frozen inputs
# and archived checkpoints without expensive retraining.
EXECUTION_PROFILE = "full"

RUN_DESCRIPTOR_REGENERATION_DIAGNOSTIC = True
RUN_FRESH_SCAFFOLD_RETRAIN_DIAGNOSTIC = False
RUN_NONAUTHORITATIVE_FULL_SCREENING_DIAGNOSTIC = False

# Use one worker for the most deterministic cross-platform retraining.
# Increase only for exploratory speed; archived/checkpoint results remain authoritative.
N_JOBS = 1

# Fresh XGBoost/scikit-learn retraining can vary slightly across operating systems
# and package builds. By default, such drift is exported and reported rather than
# stopping the notebook. Set True only inside a fully pinned historical environment.
STRICT_FRESH_RETRAINING_MATCH = False
FRESH_RETRAINING_MATCH_TOLERANCE = 1e-12

# Critical historical inputs must match their audited SHA256 hashes.
STRICT_CRITICAL_FILE_HASHES = True


Notebook release: AURKB-QSAR-VSCODE-DRIFT-SAFE-v2


In [3]:
# QSAR-VSCODE-BOOTSTRAP — Install missing packages into the active VS Code kernel
#
# This cell intentionally runs before NumPy/Pandas/RDKit/scikit-learn imports.
# It changes only the selected Python environment; it does not modify any
# project archive, historical output, or scientific result file.

import sys
import subprocess
import importlib
import importlib.util
from pathlib import Path

VALIDATED_PACKAGE_SPECS = {
    "numpy": "numpy==2.3.5",
    "pandas": "pandas==2.2.3",
    "sklearn": "scikit-learn==1.8.0",
    "xgboost": "xgboost==3.1.3",
    "rdkit": "rdkit==2025.9.4",
    "joblib": "joblib==1.5.3",
    "matplotlib": "matplotlib==3.10.8",
}

def _missing_imports():
    return [
        module_name
        for module_name in VALIDATED_PACKAGE_SPECS
        if importlib.util.find_spec(module_name) is None
    ]

missing_before = _missing_imports()

if missing_before:
    print("Missing Python modules:", ", ".join(missing_before))
    print("Active kernel interpreter:", sys.executable)

    if not bool(globals().get("AUTO_INSTALL_MISSING_DEPENDENCIES", True)):
        required = [VALIDATED_PACKAGE_SPECS[name] for name in missing_before]
        raise ModuleNotFoundError(
            "Required packages are missing from the selected VS Code kernel. "
            "Set AUTO_INSTALL_MISSING_DEPENDENCIES=True or install: "
            + " ".join(required)
        )

    configured_root = globals().get("PROJECT_ROOT")
    search_roots = [
        Path.cwd(),
        Path.cwd() / "QSAR_VSCode_Corrected",
        Path.cwd() / "QSAR_VSCode_Compatible",
    ]
    if configured_root:
        search_roots.insert(0, Path(str(configured_root)).expanduser())

    requirement_candidates = [
        root / "requirements-vscode.txt"
        for root in search_roots
    ]
    requirement_candidates.extend(Path.cwd().glob("*/requirements-vscode.txt"))
    requirements_file = next(
        (path for path in requirement_candidates if path.is_file()),
        None,
    )

    if bool(globals().get("PREFER_LOCAL_REQUIREMENTS_FILE", True)) and requirements_file:
        install_command = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-r",
            str(requirements_file),
        ]
        print("Installing from:", requirements_file.resolve())
    else:
        # Install the full validated stack, rather than only the first missing
        # import, so the kernel receives a coherent scientific environment.
        install_command = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            *VALIDATED_PACKAGE_SPECS.values(),
        ]
        print("Installing the embedded validated QSAR dependency set.")

    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "--version"],
            stdout=subprocess.DEVNULL,
        )
    except subprocess.CalledProcessError:
        subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])

    subprocess.check_call(install_command)
    importlib.invalidate_caches()

    missing_after = _missing_imports()
    if missing_after:
        raise RuntimeError(
            "Package installation completed, but these imports are still "
            f"unavailable: {missing_after}. Restart the VS Code kernel, select "
            "the same interpreter, and run from the first cell."
        )

    print("Dependency installation completed successfully.")
    print("Continue with the next cell. If VS Code reports a binary-import error, restart this kernel once.")
else:
    print("All required QSAR packages are already available.")
    print("Active kernel interpreter:", sys.executable)


All required QSAR packages are already available.
Active kernel interpreter: c:\Users\Prottoy\miniconda3\envs\research-py312\python.exe


In [4]:
# QSAR-01 — Scientific imports, deterministic configuration, isolated output tree, and execution logging
import os
import sys
import json
import math
import uuid
import time
import random
import shutil
import zipfile
import hashlib
import platform
import warnings
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, permutation_test_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, confusion_matrix, brier_score_loss,
    precision_score, recall_score, f1_score, make_scorer,
    roc_curve, precision_recall_curve,
)
import sklearn
import xgboost as xgb

from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def _env_bool(name, default):
    value = os.environ.get(name)
    if value is None:
        return bool(default)
    return value.strip().lower() not in {"0", "false", "no", "off"}

# Values from QSAR-VSCODE-CONFIG take precedence. Environment variables remain
# supported for command-line or automated execution.
EXECUTION_PROFILE = str(globals().get(
    "EXECUTION_PROFILE",
    os.environ.get("QSAR_EXECUTION_PROFILE", "full"),
)).strip().lower()
if EXECUTION_PROFILE not in {"full", "checkpoint_replay"}:
    raise ValueError("EXECUTION_PROFILE must be 'full' or 'checkpoint_replay'.")

RUN_DESCRIPTOR_REGENERATION_DIAGNOSTIC = bool(globals().get(
    "RUN_DESCRIPTOR_REGENERATION_DIAGNOSTIC",
    _env_bool("QSAR_DESCRIPTOR_DIAGNOSTIC", True),
))
RUN_FRESH_SCAFFOLD_RETRAIN_DIAGNOSTIC = bool(globals().get(
    "RUN_FRESH_SCAFFOLD_RETRAIN_DIAGNOSTIC",
    _env_bool("QSAR_FRESH_SCAFFOLD_RETRAIN", False),
))
RUN_NONAUTHORITATIVE_FULL_SCREENING_DIAGNOSTIC = bool(globals().get(
    "RUN_NONAUTHORITATIVE_FULL_SCREENING_DIAGNOSTIC",
    _env_bool("QSAR_FULL_SCREENING_DIAGNOSTIC", False),
))
N_JOBS = int(globals().get("N_JOBS", os.environ.get("QSAR_N_JOBS", "-1")))
STRICT_CRITICAL_FILE_HASHES = bool(globals().get("STRICT_CRITICAL_FILE_HASHES", True))
STRICT_FRESH_RETRAINING_MATCH = bool(globals().get("STRICT_FRESH_RETRAINING_MATCH", False))
FRESH_RETRAINING_MATCH_TOLERANCE = float(globals().get("FRESH_RETRAINING_MATCH_TOLERANCE", 1e-12))

def _normalise_optional_path(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    return Path(text).expanduser().resolve()

def _required_archives_present(root):
    root = Path(root)
    locked_present = any((root / name).is_file() for name in [
        "LOCKED_QSAR_GCN_RESULTS_DO_NOT_OVERWRITE(1).zip",
        "LOCKED_QSAR_GCN_RESULTS_DO_NOT_OVERWRITE.zip",
    ])
    scaffold_present = (root / "Fixing_scafold_validation.zip").is_file()
    return locked_present and scaffold_present

def _resolve_project_root():
    configured = _normalise_optional_path(globals().get("PROJECT_ROOT"))
    if configured is None:
        configured = _normalise_optional_path(os.environ.get("QSAR_PROJECT_ROOT"))
    if configured is not None:
        if not configured.is_dir():
            raise FileNotFoundError(f"Configured PROJECT_ROOT does not exist: {configured}")
        return configured

    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for base in list(candidates):
        for name in ("Inputs", "Input", "Archives", "Data"):
            candidate = base / name
            if candidate.is_dir():
                candidates.append(candidate)

    for candidate in candidates:
        if _required_archives_present(candidate):
            return candidate

    return cwd

BASE_DIR = _resolve_project_root()
USER_INPUT_ARCHIVE_DIR = _normalise_optional_path(globals().get("INPUT_ARCHIVE_DIR"))
if USER_INPUT_ARCHIVE_DIR is None:
    USER_INPUT_ARCHIVE_DIR = _normalise_optional_path(os.environ.get("QSAR_INPUT_ARCHIVE_DIR"))
INPUT_ARCHIVE_DIR = USER_INPUT_ARCHIVE_DIR or BASE_DIR
if not INPUT_ARCHIVE_DIR.is_dir():
    raise FileNotFoundError(f"Configured INPUT_ARCHIVE_DIR does not exist: {INPUT_ARCHIVE_DIR}")
OUTPUT_ROOT = BASE_DIR / "QSAR_Final_Reproduction"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_STARTED_UTC = datetime.now(timezone.utc)
RUN_ID = RUN_STARTED_UTC.strftime("run_%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=False, exist_ok=False)

SUBDIR_NAMES = [
    "Figures", "Tables", "CSV", "Models", "Validation",
    "Applicability_Domain", "Calibration", "Y_Randomization", "Logs",
    "Manuscript_Output", "Supplementary_Output", "Provenance",
]
DIR = {name: RUN_DIR / name for name in SUBDIR_NAMES}
for path in DIR.values():
    path.mkdir(parents=True, exist_ok=False)
(DIR["Provenance"] / "Input_Cache").mkdir()
(DIR["Models"] / "Historical_Archived").mkdir()
(DIR["Models"] / "Portable_Production_Bundle").mkdir()
(DIR["Models"] / "Portable_Scaffold_Bundle").mkdir()
(DIR["Validation"] / "Diagnostics").mkdir()
(DIR["Validation"] / "NonAuthoritative_Full_Screening_Replay").mkdir()
(DIR["CSV"] / "Archived_Historical_Screening").mkdir()

EXECUTION_EVENTS = []
EXPORT_REGISTRY = []
SOURCE_ARCHIVE_STATS = {}

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def log_event(section, event, details=None):
    record = {"timestamp_utc": utc_now(), "section": section, "event": event, "details": details or {}}
    EXECUTION_EVENTS.append(record)
    print(f"[{record['timestamp_utc']}] {section}: {event}")

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def register_export(path, section, provenance_class, description):
    path = Path(path).resolve()
    if RUN_DIR.resolve() not in path.parents and path != RUN_DIR.resolve():
        raise RuntimeError(f"Output escaped isolated run directory: {path}")
    EXPORT_REGISTRY.append({
        "relative_path": str(path.relative_to(RUN_DIR)),
        "source_section": section,
        "provenance_class": provenance_class,
        "description": description,
    })
    return path

def write_json(obj, path, section, provenance_class="regenerated", description="JSON output"):
    path = register_export(path, section, provenance_class, description)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    return path

def write_csv(df, path, section, provenance_class="regenerated", description="CSV output"):
    path = register_export(path, section, provenance_class, description)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return path

def copy_source(src, dest, section, provenance_class="archived_copy", description="Read-only source copy"):
    src, dest = Path(src), Path(dest)
    dest = register_export(dest, section, provenance_class, description)
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dest)
    return dest

def save_figure(fig, path, section, description, dpi=300):
    path = register_export(path, section, "regenerated", description)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return path

CONFIG = {
    "execution_profile": EXECUTION_PROFILE,
    "seed": SEED,
    "n_jobs": N_JOBS,
    "descriptor_regeneration_diagnostic": RUN_DESCRIPTOR_REGENERATION_DIAGNOSTIC,
    "fresh_scaffold_retrain_diagnostic": RUN_FRESH_SCAFFOLD_RETRAIN_DIAGNOSTIC,
    "full_screening_diagnostic": RUN_NONAUTHORITATIVE_FULL_SCREENING_DIAGNOSTIC,
    "strict_fresh_retraining_match": STRICT_FRESH_RETRAINING_MATCH,
    "fresh_retraining_match_tolerance": FRESH_RETRAINING_MATCH_TOLERANCE,
    "output_root": str(OUTPUT_ROOT),
    "run_directory": str(RUN_DIR),
    "integrity_policy": "All outputs remain under this unique run directory; originals are read-only.",
}
write_json(CONFIG, DIR["Provenance"] / "run_configuration.json", "QSAR-01", description="Run configuration and safety controls")
log_event("QSAR-01", "Initialized isolated output directory", {"run_dir": str(RUN_DIR), "profile": EXECUTION_PROFILE})
print("Authoritative output directory:", RUN_DIR)


[2026-07-15T12:48:27.316881+00:00] QSAR-01: Initialized isolated output directory
Authoritative output directory: C:\Users\Prottoy\Downloads\Notebook_2_Retrail\Inputs\QSAR_Final_Reproduction\run_20260715T124827Z_79cebaa2


In [5]:
# QSAR-02 — Locate archives, selectively extract critical inputs, and verify SHA256 provenance
#
# Archive discovery is deliberately content-aware. The historical ZIP files may
# be renamed, but they are accepted only when their internal members match the
# audited locked-results or scaffold-validation structures.

LOCKED_REQUIRED_MEMBER_SUFFIXES = (
    "02_QSAR_ML/tables/curated_AURKB_training_data.csv",
    "02_QSAR_ML/tables/curated_training_rdkit_descriptors_raw.csv",
    "02_QSAR_ML/model_artifacts/AURKB_QSAR_production_calibrated_model.joblib",
)
SCAFFOLD_REQUIRED_MEMBER_SUFFIXES = (
    "Fixing_scafold_validation/tables/scaffold_qsar_predictions.csv",
    "Fixing_scafold_validation/artifacts/scaffold_validation_train_only_qsar_pipeline.joblib",
    "Fixing_scafold_validation/artifacts/scaffold_validation_sigmoid_calibrator.joblib",
)
MANUSCRIPT_REQUIRED_MEMBER_SUFFIXES = (
    "Manuscript",
)


def _zip_contains_suffixes(path, required_suffixes):
    """Return True only when a readable ZIP contains every required suffix."""
    try:
        path = Path(path).expanduser().resolve()
        if not path.is_file() or path.suffix.lower() != ".zip":
            return False
        with zipfile.ZipFile(path) as zf:
            names = tuple(name.replace("\\", "/") for name in zf.namelist())
        return all(any(name.endswith(suffix) for name in names) for suffix in required_suffixes)
    except (OSError, zipfile.BadZipFile, PermissionError):
        return False


def _vscode_notebook_parent():
    """Best-effort discovery of the notebook directory in VS Code Jupyter."""
    candidates = [
        globals().get("__vsc_ipynb_file__"),
        os.environ.get("VSCODE_NOTEBOOK_FILE"),
        os.environ.get("JPY_SESSION_NAME"),
    ]
    for value in candidates:
        if not value:
            continue
        try:
            path = Path(str(value)).expanduser()
            if path.suffix.lower() == ".ipynb":
                path = path.parent
            path = path.resolve()
            if path.is_dir():
                return path
        except OSError:
            pass
    return None


def _dedupe_existing_dirs(values):
    result, seen = [], set()
    for value in values:
        if value is None:
            continue
        try:
            path = Path(str(value)).expanduser().resolve()
        except OSError:
            continue
        if not path.is_dir():
            continue
        key = os.path.normcase(str(path))
        if key not in seen:
            seen.add(key)
            result.append(path)
    return result


def _archive_search_roots():
    cwd = Path.cwd().resolve()
    home = Path.home().resolve()
    configured_roots = globals().get("ARCHIVE_SEARCH_ROOTS", []) or []
    if isinstance(configured_roots, (str, Path)):
        configured_roots = [configured_roots]

    roots = [
        _normalise_optional_path(globals().get("INPUT_ARCHIVE_DIR")),
        _normalise_optional_path(globals().get("PROJECT_ROOT")),
        _vscode_notebook_parent(),
        BASE_DIR,
        INPUT_ARCHIVE_DIR,
        cwd,
        *cwd.parents,
        *configured_roots,
        home / "Downloads",
        home / "Desktop",
        home / "Documents",
        home / "OneDrive",
        home / "OneDrive" / "Downloads",
        home / "OneDrive" / "Desktop",
        home / "OneDrive" / "Documents",
    ]
    roots = _dedupe_existing_dirs(roots)

    expanded = list(roots)
    for base in roots:
        for name in ("Inputs", "Input", "Archives", "Archive", "Data", "data", "QSAR", "AURKB"):
            child = base / name
            if child.is_dir():
                expanded.append(child)
    return _dedupe_existing_dirs(expanded)


def _explicit_candidate(value, required_suffixes, label):
    path = _normalise_optional_path(value)
    if path is None:
        return None
    if path.is_dir():
        raise FileNotFoundError(
            f"{label} was set to a folder, not a ZIP file: {path}. "
            "Set the exact ZIP path or use INPUT_ARCHIVE_DIR/ARCHIVE_SEARCH_ROOTS for folders."
        )
    if not path.is_file():
        raise FileNotFoundError(f"Configured {label} does not exist: {path}")
    if not _zip_contains_suffixes(path, required_suffixes):
        raise RuntimeError(
            f"Configured {label} is not the expected historical archive: {path}. "
            "Its internal ZIP structure did not pass validation."
        )
    return path


def _search_archive(required_suffixes, preferred_names, roots, recursive=True):
    """Find a ZIP by exact/fuzzy filename, then verify its internal members."""
    examined = set()

    def consider(path):
        try:
            path = Path(path).resolve()
        except OSError:
            return None
        key = os.path.normcase(str(path))
        if key in examined:
            return None
        examined.add(key)
        return path if _zip_contains_suffixes(path, required_suffixes) else None

    # Fast exact-name search first.
    for root in roots:
        for name in preferred_names:
            found = consider(root / name)
            if found:
                return found, examined

    # Then inspect plausible ZIP filenames. The content check remains decisive.
    keyword_sets = [
        ("locked", "qsar"),
        ("scaf", "validation"),
        ("manuscript",),
    ]
    preferred_lower = " ".join(preferred_names).lower()
    keywords = next((ks for ks in keyword_sets if all(k in preferred_lower for k in ks)), ())

    for root in roots:
        try:
            iterator = root.rglob("*.zip") if recursive else root.glob("*.zip")
            for path in iterator:
                # Avoid traversing environments and generated outputs.
                lower_parts = {part.lower() for part in path.parts}
                if lower_parts.intersection({".venv", "venv", "site-packages", "node_modules", "qsar_final_reproduction"}):
                    continue
                name_lower = path.name.lower()
                if keywords and not all(k in name_lower for k in keywords):
                    # Still allow content validation for ZIPs directly in a root,
                    # but skip unrelated deep files to keep the search bounded.
                    try:
                        if path.parent.resolve() != root.resolve():
                            continue
                    except OSError:
                        continue
                found = consider(path)
                if found:
                    return found, examined
                if len(examined) >= 2500:
                    return None, examined
        except (OSError, PermissionError):
            continue
    return None, examined


def _pick_zip_with_dialog(title, required_suffixes):
    """Use a native desktop dialog; return None when GUI support is unavailable."""
    if not bool(globals().get("ENABLE_INTERACTIVE_ARCHIVE_PICKER", True)):
        return None
    try:
        import tkinter as tk
        from tkinter import filedialog, messagebox
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        selected = filedialog.askopenfilename(
            title=title,
            filetypes=[("ZIP archives", "*.zip"), ("All files", "*.*")],
            initialdir=str(Path.home() / "Downloads") if (Path.home() / "Downloads").is_dir() else str(Path.home()),
        )
        root.destroy()
        if not selected:
            return None
        path = Path(selected).expanduser().resolve()
        if not _zip_contains_suffixes(path, required_suffixes):
            try:
                messagebox.showerror(
                    "Incorrect archive",
                    f"The selected file is not the expected historical archive:\n{path}",
                )
            except Exception:
                pass
            raise RuntimeError(
                f"Selected ZIP does not contain the required audited members: {path}"
            )
        return path
    except (ImportError, OSError, RuntimeError) as exc:
        print(f"Interactive archive picker unavailable or unsuccessful: {exc}")
        return None


SEARCH_ROOTS = _archive_search_roots()
recursive_search = bool(globals().get("RECURSIVE_ARCHIVE_SEARCH", True))

LOCKED_ZIP = _explicit_candidate(
    globals().get("LOCKED_ZIP_PATH"),
    LOCKED_REQUIRED_MEMBER_SUFFIXES,
    "LOCKED_ZIP_PATH",
)
SCAFFOLD_ZIP = _explicit_candidate(
    globals().get("SCAFFOLD_ZIP_PATH"),
    SCAFFOLD_REQUIRED_MEMBER_SUFFIXES,
    "SCAFFOLD_ZIP_PATH",
)
MANUSCRIPT_ZIP = _normalise_optional_path(globals().get("MANUSCRIPT_ZIP_PATH"))
if MANUSCRIPT_ZIP is not None and not MANUSCRIPT_ZIP.is_file():
    raise FileNotFoundError(f"Configured MANUSCRIPT_ZIP_PATH does not exist: {MANUSCRIPT_ZIP}")

searched_locked = searched_scaffold = set()
if LOCKED_ZIP is None:
    LOCKED_ZIP, searched_locked = _search_archive(
        LOCKED_REQUIRED_MEMBER_SUFFIXES,
        [
            "LOCKED_QSAR_GCN_RESULTS_DO_NOT_OVERWRITE(1).zip",
            "LOCKED_QSAR_GCN_RESULTS_DO_NOT_OVERWRITE.zip",
        ],
        SEARCH_ROOTS,
        recursive=recursive_search,
    )
if SCAFFOLD_ZIP is None:
    SCAFFOLD_ZIP, searched_scaffold = _search_archive(
        SCAFFOLD_REQUIRED_MEMBER_SUFFIXES,
        ["Fixing_scafold_validation.zip", "Fixing_scaffold_validation.zip"],
        SEARCH_ROOTS,
        recursive=recursive_search,
    )

# A local VS Code desktop session can browse to archives located anywhere.
if LOCKED_ZIP is None:
    print("Locked-results archive was not found automatically. Select it in the file dialog.")
    LOCKED_ZIP = _pick_zip_with_dialog(
        "Select LOCKED_QSAR_GCN_RESULTS_DO_NOT_OVERWRITE ZIP",
        LOCKED_REQUIRED_MEMBER_SUFFIXES,
    )
if SCAFFOLD_ZIP is None:
    print("Scaffold-validation archive was not found automatically. Select it in the file dialog.")
    SCAFFOLD_ZIP = _pick_zip_with_dialog(
        "Select Fixing_scafold_validation ZIP",
        SCAFFOLD_REQUIRED_MEMBER_SUFFIXES,
    )

if LOCKED_ZIP is None or SCAFFOLD_ZIP is None:
    searched_folders = [str(path) for path in SEARCH_ROOTS]
    missing = []
    if LOCKED_ZIP is None:
        missing.append("locked QSAR/GCN results ZIP")
    if SCAFFOLD_ZIP is None:
        missing.append("scaffold-validation ZIP")
    raise FileNotFoundError(
        "Could not locate: " + ", ".join(missing) + ".\n\n"
        "Fix this in QSAR-VSCODE-CONFIG by setting exact raw-string paths, for example:\n"
        'LOCKED_ZIP_PATH = r"C:\\full\\path\\LOCKED_QSAR_GCN_RESULTS_DO_NOT_OVERWRITE(1).zip"\n'
        'SCAFFOLD_ZIP_PATH = r"C:\\full\\path\\Fixing_scafold_validation.zip"\n\n'
        "Alternatively, place both ZIP files beside the notebook or inside an Inputs folder.\n"
        f"Searched roots: {searched_folders}"
    )

print("Validated locked-results archive:", LOCKED_ZIP)
print("Validated scaffold-validation archive:", SCAFFOLD_ZIP)
if MANUSCRIPT_ZIP:
    print("Optional manuscript archive:", MANUSCRIPT_ZIP)

for archive in [LOCKED_ZIP, SCAFFOLD_ZIP] + ([MANUSCRIPT_ZIP] if MANUSCRIPT_ZIP else []):
    SOURCE_ARCHIVE_STATS[str(archive)] = {
        "size_bytes": archive.stat().st_size,
        "mtime_ns": archive.stat().st_mtime_ns,
        "sha256": sha256_file(archive),
    }

CACHE = DIR["Provenance"] / "Input_Cache"

def extract_unique_suffix(zip_path, suffix, destination_name, category):
    destination = CACHE / category / destination_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        matches = [name for name in zf.namelist() if name.endswith(suffix)]
        if len(matches) != 1:
            raise RuntimeError(f"Expected one ZIP member ending {suffix!r}; found {len(matches)}: {matches[:10]}")
        member = matches[0]
        with zf.open(member) as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target)
    return destination, member

LOCKED_MEMBERS = {
    "raw": ("02_QSAR_ML/tables/CHEMBL2185_IC50_ChEMBL_raw.csv", "CHEMBL2185_IC50_ChEMBL_raw.csv"),
    "curated": ("02_QSAR_ML/tables/curated_AURKB_training_data.csv", "curated_AURKB_training_data.csv"),
    "descriptors": ("02_QSAR_ML/tables/curated_training_rdkit_descriptors_raw.csv", "curated_training_rdkit_descriptors_raw.csv"),
    "curation_log": ("02_QSAR_ML/tables/curation_log.csv", "curation_log.csv"),
    "cv_archived": ("02_QSAR_ML/tables/model_comparison_cross_validation.csv", "model_comparison_cross_validation.csv"),
    "selected_descriptors": ("02_QSAR_ML/tables/selected_rdkit_descriptors.csv", "selected_rdkit_descriptors.csv"),
    "y_scores_archived": ("02_QSAR_ML/tables/y_randomization_permutation_scores.csv", "y_randomization_permutation_scores.csv"),
    "y_summary_archived": ("02_QSAR_ML/tables/y_randomization_summary.json", "y_randomization_summary.json"),
    "run_config": ("02_QSAR_ML/run_config.json", "run_config.json"),
    "artifact_summary": ("02_QSAR_ML/model_artifacts/artifact_summary.json", "artifact_summary.json"),
    "production_model": ("02_QSAR_ML/model_artifacts/AURKB_QSAR_production_calibrated_model.joblib", "AURKB_QSAR_production_calibrated_model.joblib"),
    "ad_preprocessor": ("02_QSAR_ML/model_artifacts/AURKB_QSAR_AD_descriptor_preprocessor.joblib", "AURKB_QSAR_AD_descriptor_preprocessor.joblib"),
    "ad_matrix": ("02_QSAR_ML/model_artifacts/AURKB_QSAR_AD_X_full_scaled.npy", "AURKB_QSAR_AD_X_full_scaled.npy"),
    "prescreen": ("AURKB_prescreen_20260520_005631/Data/Prescreening_coconut_AURKB.csv", "Prescreening_coconut_AURKB.csv"),
    "candidate_snapshot": (
        "08_All_Raw_Output_Backup/03_DeepChem_GCN_Consensus_20260521_074502/tables/gcn_candidate_input_audit.csv",
        "gcn_candidate_input_audit.csv",
    ),
}
SCAFFOLD_MEMBERS = {
    "assignment": ("Fixing_scafold_validation/tables/scaffold_compound_split_assignment.csv", "scaffold_compound_split_assignment.csv"),
    "metrics": ("Fixing_scafold_validation/tables/scaffold_qsar_metrics.csv", "scaffold_qsar_metrics.csv"),
    "predictions": ("Fixing_scafold_validation/tables/scaffold_qsar_predictions.csv", "scaffold_qsar_predictions.csv"),
    "split_summary": ("Fixing_scafold_validation/tables/scaffold_split_summary.csv", "scaffold_split_summary.csv"),
    "overlap": ("Fixing_scafold_validation/tables/scaffold_split_overlap_summary.json", "scaffold_split_overlap_summary.json"),
    "ad_summary": ("Fixing_scafold_validation/tables/scaffold_qsar_applicability_domain_summary.csv", "scaffold_qsar_applicability_domain_summary.csv"),
    "confusion": ("Fixing_scafold_validation/tables/scaffold_qsar_confusion_matrix.csv", "scaffold_qsar_confusion_matrix.csv"),
    "selected": ("Fixing_scafold_validation/tables/scaffold_validation_train_only_selected_descriptors.csv", "scaffold_validation_train_only_selected_descriptors.csv"),
    "pipeline": ("Fixing_scafold_validation/artifacts/scaffold_validation_train_only_qsar_pipeline.joblib", "scaffold_validation_train_only_qsar_pipeline.joblib"),
    "calibrator": ("Fixing_scafold_validation/artifacts/scaffold_validation_sigmoid_calibrator.joblib", "scaffold_validation_sigmoid_calibrator.joblib"),
    "config": ("Fixing_scafold_validation/scaffold_validation_config.json", "scaffold_validation_config.json"),
    "readme": ("Fixing_scafold_validation/scaffold_validation_README.txt", "scaffold_validation_README.txt"),
}

INPUTS = {}
SOURCE_MEMBER_MANIFEST = []
for key, (suffix, filename) in LOCKED_MEMBERS.items():
    path, member = extract_unique_suffix(LOCKED_ZIP, suffix, filename, "locked")
    INPUTS[key] = path
    SOURCE_MEMBER_MANIFEST.append({"key": key, "archive": str(LOCKED_ZIP), "member": member, "local_cache": str(path), "sha256": sha256_file(path)})
for key, (suffix, filename) in SCAFFOLD_MEMBERS.items():
    path, member = extract_unique_suffix(SCAFFOLD_ZIP, suffix, filename, "scaffold")
    INPUTS[f"scaffold_{key}"] = path
    SOURCE_MEMBER_MANIFEST.append({"key": f"scaffold_{key}", "archive": str(SCAFFOLD_ZIP), "member": member, "local_cache": str(path), "sha256": sha256_file(path)})

EXPECTED_CRITICAL_HASHES = {
    "raw": "84e754ae9983745b88eb95964fd617b8febe9ebaf47f5ac293e3f0b22dee5b1e",
    "curated": "8c5c9380b54d51eb485063d603fc815e7a6604e4daacbeabd29323e625688192",
    "descriptors": "1696615608245b56845d7f77e3e1c9566bbbd1f8523f6cad7d92903e8cb175f7",
    "cv_archived": "3e861581d5e371886e9419e654fb57c73b01823f017bd6bebc7e56657ec4c392",
    "y_scores_archived": "61d4e4d6e1199498a003c4bbe05707a6e08349beb32e7485c55370d6605a23f9",
    "production_model": "b6674271e58fe1189d04de14b6a4d1884a13ef051d9d19af75acd56843a29930",
    "ad_preprocessor": "83b53175d26b949ea8c5060c336aff7c01908f67b32150ae0c465510aa35335d",
    "ad_matrix": "08ffdbfe72633c4c8d2d62574fd46269aa9e2f8a7a3b21e6537d3a09e59d057b",
    "candidate_snapshot": "8821947e92d8b209d291b627bae3b1d5068c6e2f1c0154f695abf0e73673f4df",
    "scaffold_assignment": "37cca909f24805b1bbe3d792576f64a701967efd72462d8ca56bce3f53084f5c",
    "scaffold_predictions": "a5fa6dfdca734cc635f29704c7fde0d8752ac73843663194342b5c12517857ae",
    "scaffold_pipeline": "8566fbe089c0bb2f626968cabad80bb1271b38c18c2931664982e096cb46dda2",
    "scaffold_calibrator": "b2cfc505fe1e67896cc69d8209e95ffa15540db72793c1052ff6161be97aeb7d",
}
HASH_CHECKS = []
for key, expected in EXPECTED_CRITICAL_HASHES.items():
    actual = sha256_file(INPUTS[key])
    matched = actual == expected
    HASH_CHECKS.append({"input_key": key, "expected_sha256": expected, "actual_sha256": actual, "matched": matched})
    if STRICT_CRITICAL_FILE_HASHES and not matched:
        raise RuntimeError(f"Critical historical input hash mismatch for {key}: {actual} != {expected}")

write_csv(pd.DataFrame(SOURCE_MEMBER_MANIFEST), DIR["Provenance"] / "source_member_manifest.csv", "QSAR-02", "provenance", "ZIP member inventory and hashes")
write_csv(pd.DataFrame(HASH_CHECKS), DIR["Provenance"] / "critical_input_hash_verification.csv", "QSAR-02", "provenance", "Strict critical-file SHA256 verification")
write_json(SOURCE_ARCHIVE_STATS, DIR["Provenance"] / "source_archive_hashes.json", "QSAR-02", "provenance", "Source archive file hashes and pre-run metadata")
log_event("QSAR-02", "Extracted and hash-verified critical historical inputs", {"n_members": len(SOURCE_MEMBER_MANIFEST), "strict": STRICT_CRITICAL_FILE_HASHES})


Validated locked-results archive: C:\Users\Prottoy\Downloads\Notebook_2_Retrail\Inputs\LOCKED_QSAR_GCN_RESULTS_DO_NOT_OVERWRITE(1).zip
Validated scaffold-validation archive: C:\Users\Prottoy\Downloads\Notebook_2_Retrail\Inputs\Fixing_scafold_validation.zip
[2026-07-15T12:48:27.957024+00:00] QSAR-02: Extracted and hash-verified critical historical inputs


In [6]:
# QSAR-03 — Shared scientific helpers, metrics, portable checkpoint serialization, and plotting
NAN_THRESHOLD = 0.20
CORR_THRESHOLD = 0.95
TEST_SIZE = 0.20
N_CV_SPLITS = 5
N_PERMUTATIONS = 30
QSAR_ACTIVE_THRESHOLD = 0.50
HIGH_CONFIDENCE_THRESHOLD = 0.75

class DescriptorPreprocessor(BaseEstimator, TransformerMixin):
    """Historical leakage-controlled descriptor preprocessor used by archived joblib objects."""
    def __init__(self, nan_threshold=0.20, corr_threshold=0.95):
        self.nan_threshold = nan_threshold
        self.corr_threshold = corr_threshold

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy().replace([np.inf, -np.inf], np.nan)
        self.input_columns_ = list(X.columns)
        nan_fraction = X.isna().mean()
        self.nan_keep_columns_ = nan_fraction[nan_fraction <= self.nan_threshold].index.tolist()
        X1 = X[self.nan_keep_columns_].copy()
        self.medians_ = X1.median(numeric_only=True)
        X1 = X1.fillna(self.medians_)
        self.variance_selector_ = VarianceThreshold(0.0).fit(X1)
        self.var_keep_columns_ = X1.columns[self.variance_selector_.get_support()].tolist()
        X2 = X1[self.var_keep_columns_].copy()
        corr = X2.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        self.corr_drop_columns_ = [col for col in upper.columns if any(upper[col] > self.corr_threshold)]
        self.feature_names_ = [col for col in X2.columns if col not in self.corr_drop_columns_]
        self.scaler_ = StandardScaler().fit(X2[self.feature_names_].values)
        self.n_features_in_ = len(self.input_columns_)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy().replace([np.inf, -np.inf], np.nan)
        X = X.reindex(columns=self.nan_keep_columns_, fill_value=np.nan).fillna(self.medians_)
        X = X.reindex(columns=self.var_keep_columns_, fill_value=0.0)
        X = X.drop(columns=[c for c in self.corr_drop_columns_ if c in X.columns], errors="ignore")
        X = X.reindex(columns=self.feature_names_, fill_value=0.0)
        return self.scaler_.transform(X.values)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_)

def metric_row(y_true, probability, threshold=0.50, label=""):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        "dataset": label,
        "n": int(len(y_true)),
        "n_active": int((y_true == 1).sum()),
        "n_inactive": int((y_true == 0).sum()),
        "roc_auc": float(roc_auc_score(y_true, probability)),
        "pr_auc": float(average_precision_score(y_true, probability)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, prediction)),
        "mcc": float(matthews_corrcoef(y_true, prediction)),
        "f1": float(f1_score(y_true, prediction)),
        "precision": float(precision_score(y_true, prediction)),
        "sensitivity": float(recall_score(y_true, prediction)),
        "specificity": float(tn / (tn + fp)),
        "brier_score": float(brier_score_loss(y_true, probability)),
        "threshold": float(threshold),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def calculate_leverage(X_reference_scaled, X_query_scaled):
    inverse = np.linalg.pinv(X_reference_scaled.T @ X_reference_scaled)
    return np.einsum("ij,jk,ik->i", X_query_scaled, inverse, X_query_scaled, optimize=True)

def h_star(n_samples, n_features):
    return 3.0 * (n_features + 1) / n_samples

def safe_logit(probability, eps=1e-6):
    probability = np.clip(np.asarray(probability, dtype=float), eps, 1 - eps)
    return np.log(probability / (1 - probability)).reshape(-1, 1)

def sigmoid_calibration(raw_probability, a, b):
    z = np.clip(a * np.asarray(raw_probability, dtype=float) + b, -700, 700)
    return 1.0 / (1.0 + np.exp(z))

def standardize_smiles(smiles):
    try:
        molecule = Chem.MolFromSmiles(str(smiles))
        if molecule is None:
            return None
        molecule = rdMolStandardize.Cleanup(molecule)
        molecule = rdMolStandardize.FragmentParent(molecule)
        molecule = rdMolStandardize.Uncharger().uncharge(molecule)
        Chem.SanitizeMol(molecule)
        return Chem.MolToSmiles(molecule, isomericSmiles=True)
    except Exception:
        return None

def smiles_to_inchikey(smiles):
    try:
        molecule = Chem.MolFromSmiles(str(smiles))
        return Chem.MolToInchiKey(molecule) if molecule is not None else None
    except Exception:
        return None

def scaffold_from_smiles(smiles):
    try:
        molecule = Chem.MolFromSmiles(str(smiles))
        if molecule is None:
            return "INVALID"
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=molecule, includeChirality=False)
        return scaffold if scaffold else Chem.MolToSmiles(molecule, isomericSmiles=False)
    except Exception:
        return "INVALID"

def replay_chembl_curation(raw):
    df = raw.copy()
    curation_steps = []
    def apply_step(label, operation):
        nonlocal df
        before = len(df)
        df = operation(df).copy()
        curation_steps.append({"step": label, "before": before, "after": len(df), "removed": before - len(df)})
    apply_step("Keep exact standard_relation '='", lambda x: x[x["standard_relation"].astype(str).str.strip().eq("=")])
    def valid_activity(x):
        x = x.dropna(subset=["canonical_smiles", "standard_value", "standard_units"]).copy()
        x["standard_value"] = pd.to_numeric(x["standard_value"], errors="coerce")
        return x.dropna(subset=["standard_value"])[lambda q: q["standard_value"] > 0]
    apply_step("Remove missing/non-positive SMILES or activity", valid_activity)
    apply_step("Keep standard_units in ['nM']", lambda x: x[x["standard_units"].isin(["nM"])])
    apply_step("Keep assay_type in ['B']", lambda x: x[x["assay_type"].isin(["B"])])
    def standardized(x):
        x = x.copy()
        x["smiles_std"] = x["canonical_smiles"].apply(standardize_smiles)
        return x.dropna(subset=["smiles_std"])
    apply_step("RDKit standardization and invalid structure removal", standardized)
    def add_inchikey(x):
        x = x.copy()
        x["inchikey_std"] = x["smiles_std"].apply(smiles_to_inchikey)
        return x.dropna(subset=["inchikey_std"])
    apply_step("Generate standardized InChIKey", add_inchikey)
    df["pActivity"] = 9.0 - np.log10(df["standard_value"].astype(float))
    grouped = df.groupby("inchikey_std", dropna=False).agg(
        molecule_chembl_id=("molecule_chembl_id", lambda x: ";".join(sorted(set(map(str, x))))),
        smiles_std=("smiles_std", "first"),
        activity_nM_median=("standard_value", "median"),
        activity_nM_min=("standard_value", "min"),
        activity_nM_max=("standard_value", "max"),
        pActivity_median=("pActivity", "median"),
        pActivity_min=("pActivity", "min"),
        pActivity_max=("pActivity", "max"),
        n_measurements=("standard_value", "count"),
        assay_chembl_ids=("assay_chembl_id", lambda x: ";".join(sorted(set(map(str, x.dropna()))))),
        document_year_min=("document_year", "min"),
        document_year_max=("document_year", "max"),
    ).reset_index()
    curation_steps.append({"step": "Deduplicate by standardized InChIKey using median activity", "before": len(df), "after": len(grouped), "removed": len(df) - len(grouped)})
    grouped["pActivity_range"] = grouped["pActivity_max"] - grouped["pActivity_min"]
    grouped["duplicate_conflict_flag"] = (grouped["n_measurements"] > 1) & (grouped["pActivity_range"] > 1.0)
    grouped["y"] = (grouped["pActivity_median"] >= 6.0).astype(int)
    grouped["activity_label"] = np.where(grouped["y"] == 1, "Active", "Inactive")
    grouped["scaffold"] = grouped["smiles_std"].apply(scaffold_from_smiles)
    return grouped, pd.DataFrame(curation_steps)

def descriptor_dataframe(smiles_series):
    names = [item[0] for item in Descriptors.descList]
    calculator = MoleculeDescriptors.MolecularDescriptorCalculator(names)
    rows = []
    for smiles in smiles_series:
        molecule = Chem.MolFromSmiles(str(smiles))
        if molecule is None:
            rows.append([np.nan] * len(names))
        else:
            try:
                rows.append(list(calculator.CalcDescriptors(molecule)))
            except Exception:
                rows.append([np.nan] * len(names))
    return pd.DataFrame(rows, columns=names).replace([np.inf, -np.inf], np.nan)

def serialize_preprocessor(preprocessor, folder, prefix):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    metadata = {
        "nan_threshold": float(preprocessor.nan_threshold),
        "corr_threshold": float(preprocessor.corr_threshold),
        "input_columns": list(preprocessor.input_columns_),
        "nan_keep_columns": list(preprocessor.nan_keep_columns_),
        "var_keep_columns": list(preprocessor.var_keep_columns_),
        "corr_drop_columns": list(preprocessor.corr_drop_columns_),
        "feature_names": list(preprocessor.feature_names_),
        "medians": {str(k): float(v) for k, v in preprocessor.medians_.items()},
    }
    metadata_path = folder / f"{prefix}_preprocessor.json"
    arrays_path = folder / f"{prefix}_preprocessor_arrays.npz"
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    np.savez_compressed(
        arrays_path,
        scaler_mean=np.asarray(preprocessor.scaler_.mean_, dtype=float),
        scaler_scale=np.asarray(preprocessor.scaler_.scale_, dtype=float),
        scaler_var=np.asarray(preprocessor.scaler_.var_, dtype=float),
    )
    return metadata, {
        "scaler_mean": np.asarray(preprocessor.scaler_.mean_, dtype=float),
        "scaler_scale": np.asarray(preprocessor.scaler_.scale_, dtype=float),
        "scaler_var": np.asarray(preprocessor.scaler_.var_, dtype=float),
    }, metadata_path, arrays_path

def portable_transform(X, metadata, arrays):
    X = pd.DataFrame(X).copy().replace([np.inf, -np.inf], np.nan)
    X = X.reindex(columns=metadata["nan_keep_columns"], fill_value=np.nan)
    X = X.fillna(pd.Series(metadata["medians"]))
    X = X.reindex(columns=metadata["var_keep_columns"], fill_value=0.0)
    X = X.drop(columns=[c for c in metadata["corr_drop_columns"] if c in X.columns], errors="ignore")
    X = X.reindex(columns=metadata["feature_names"], fill_value=0.0)
    return (X.to_numpy(dtype=float) - arrays["scaler_mean"]) / arrays["scaler_scale"]

def load_joblib_with_warnings(path):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        obj = joblib.load(path)
    return obj, [str(item.message) for item in caught]

def compare_numeric(actual, expected, tolerance=1e-12, label="value"):
    difference = abs(float(actual) - float(expected))
    if difference > tolerance:
        raise AssertionError(f"{label}: actual={actual}, expected={expected}, difference={difference}")
    return difference

def plot_confusion(y_true, probability, threshold, title, path, section):
    cm = confusion_matrix(y_true, (np.asarray(probability) >= threshold).astype(int), labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    image = ax.imshow(cm)
    for (row, col), value in np.ndenumerate(cm):
        ax.text(col, row, str(value), ha="center", va="center")
    ax.set_xticks([0, 1], labels=["Inactive", "Active"])
    ax.set_yticks([0, 1], labels=["Inactive", "Active"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Observed")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    save_figure(fig, path, section, title)

def plot_roc_pr_calibration(y_true, probability, prefix, title_prefix, section):
    fpr, tpr, _ = roc_curve(y_true, probability)
    precision, recall, _ = precision_recall_curve(y_true, probability)
    observed, predicted = calibration_curve(y_true, probability, n_bins=10, strategy="quantile")
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    ax.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_true, probability):.4f}")
    ax.plot([0, 1], [0, 1], "--")
    ax.set(xlabel="False-positive rate", ylabel="True-positive rate", title=f"{title_prefix} ROC")
    ax.legend(); ax.grid(alpha=0.25); fig.tight_layout()
    save_figure(fig, DIR["Figures"] / f"{prefix}_roc_curve.png", section, f"{title_prefix} ROC curve")
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    ax.plot(recall, precision, label=f"PR-AUC = {average_precision_score(y_true, probability):.4f}")
    ax.set(xlabel="Recall", ylabel="Precision", title=f"{title_prefix} precision-recall")
    ax.legend(); ax.grid(alpha=0.25); fig.tight_layout()
    save_figure(fig, DIR["Figures"] / f"{prefix}_precision_recall_curve.png", section, f"{title_prefix} precision-recall curve")
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    ax.plot(predicted, observed, "o-")
    ax.plot([0, 1], [0, 1], "--")
    ax.set(xlabel="Mean predicted probability", ylabel="Observed active fraction", title=f"{title_prefix} calibration")
    ax.grid(alpha=0.25); fig.tight_layout()
    save_figure(fig, DIR["Calibration"] / f"{prefix}_calibration_curve.png", section, f"{title_prefix} calibration curve")

log_event("QSAR-03", "Defined scientific and provenance helpers")


[2026-07-15T12:48:27.993428+00:00] QSAR-03: Defined scientific and provenance helpers


In [7]:
# QSAR-04 — Load authoritative frozen inputs and reproduce the curation sequence
RAW = pd.read_csv(INPUTS["raw"])
CURATED_FROZEN = pd.read_csv(INPUTS["curated"])
X_FROZEN = pd.read_csv(INPUTS["descriptors"]).replace([np.inf, -np.inf], np.nan)
CURATION_LOG_ARCHIVED = pd.read_csv(INPUTS["curation_log"])
Y = CURATED_FROZEN["y"].astype(int).to_numpy()

assert RAW.shape[0] == 3248
assert CURATED_FROZEN.shape[0] == 1854
assert X_FROZEN.shape == (1854, 217)
assert int((Y == 1).sum()) == 1429 and int((Y == 0).sum()) == 425
assert CURATED_FROZEN["inchikey_std"].duplicated().sum() == 0

INPUT_AUDIT = {
    "raw_records": int(len(RAW)),
    "curated_compounds": int(len(CURATED_FROZEN)),
    "active": int((Y == 1).sum()),
    "inactive": int((Y == 0).sum()),
    "median_pIC50": float(CURATED_FROZEN["pActivity_median"].median()),
    "unique_scaffolds": int(CURATED_FROZEN["scaffold"].nunique()),
    "raw_descriptor_rows": int(X_FROZEN.shape[0]),
    "raw_descriptor_columns": int(X_FROZEN.shape[1]),
}
write_json(INPUT_AUDIT, DIR["Tables"] / "QSAR_input_audit.json", "QSAR-04", description="Recomputed frozen-input audit")
copy_source(INPUTS["raw"], DIR["CSV"] / "CHEMBL2185_IC50_ChEMBL_raw_authoritative.csv", "QSAR-04", description="Hash-verified frozen ChEMBL raw table")
copy_source(INPUTS["curated"], DIR["CSV"] / "curated_AURKB_training_data_authoritative.csv", "QSAR-04", description="Hash-verified frozen curated training table")
copy_source(INPUTS["curation_log"], DIR["Supplementary_Output"] / "QSAR_curation_log_archived.csv", "QSAR-04", description="Archived curation step log")

CURATED_REPLAY, CURATION_LOG_REPLAY = replay_chembl_curation(RAW)
write_csv(CURATED_REPLAY, DIR["Validation"] / "curation_replay_current_environment.csv", "QSAR-04", "diagnostic", "Current-environment curation replay")
write_csv(CURATION_LOG_REPLAY, DIR["Validation"] / "curation_log_replayed.csv", "QSAR-04", "regenerated", "Executable curation step counts")

aligned = CURATED_FROZEN.merge(CURATED_REPLAY, on="inchikey_std", how="outer", suffixes=("_frozen", "_replay"), indicator=True, validate="one_to_one")
CURATION_COMPARISON = {
    "same_inchikey_set": bool((aligned["_merge"] == "both").all()),
    "n_frozen": int(len(CURATED_FROZEN)),
    "n_replay": int(len(CURATED_REPLAY)),
    "label_mismatches": int((aligned["y_frozen"] != aligned["y_replay"]).sum()),
    "activity_value_mismatches": int((~np.isclose(aligned["pActivity_median_frozen"], aligned["pActivity_median_replay"], rtol=0, atol=1e-12)).sum()),
    "scaffold_mismatches": int((aligned["scaffold_frozen"].fillna("") != aligned["scaffold_replay"].fillna("")).sum()),
    "standardized_smiles_text_mismatches": int((aligned["smiles_std_frozen"].fillna("") != aligned["smiles_std_replay"].fillna("")).sum()),
    "interpretation": "Molecular identities, labels, activities, and scaffolds reproduce exactly; a small number of canonical SMILES text encodings may differ by RDKit version.",
}
assert CURATION_COMPARISON["same_inchikey_set"]
assert CURATION_COMPARISON["label_mismatches"] == 0
assert CURATION_COMPARISON["activity_value_mismatches"] == 0
assert CURATION_COMPARISON["scaffold_mismatches"] == 0
write_json(CURATION_COMPARISON, DIR["Validation"] / "curation_replay_comparison.json", "QSAR-04", "regenerated", "Frozen-versus-replayed curation comparison")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(CURATED_FROZEN["pActivity_median"], bins=35, edgecolor="white", linewidth=0.5)
axes[0].axvline(6.0, linestyle="--", linewidth=2, label="1,000 nM cutoff (pIC50 6.0)")
axes[0].set(xlabel="Median pIC50", ylabel="Compounds", title="Activity distribution"); axes[0].legend()
class_counts = CURATED_FROZEN["activity_label"].value_counts().reindex(["Active", "Inactive"])
axes[1].bar(class_counts.index, class_counts.values); axes[1].set(ylabel="Compounds", title="Class balance")
for position, value in enumerate(class_counts.values): axes[1].text(position, value, str(value), ha="center", va="bottom")
scaffold_sizes = CURATED_FROZEN["scaffold"].value_counts()
axes[2].hist(scaffold_sizes.values, bins=30, edgecolor="white", linewidth=0.5)
axes[2].set(xlabel="Compounds per scaffold", ylabel="Scaffolds", title="Scaffold-size distribution")
fig.suptitle("Curated human AURKB IC50 dataset"); fig.tight_layout()
save_figure(fig, DIR["Figures"] / "01_curated_dataset_summary.png", "QSAR-04", "Curated AURKB training-dataset summary")
log_event("QSAR-04", "Reproduced curation counts and molecular identities", CURATION_COMPARISON)


[18:48:28] Initializing MetalDisconnector
[18:48:28] Running MetalDisconnector
[18:48:28] Initializing Normalizer
[18:48:28] Running Normalizer
[18:48:28] Initializing MetalDisconnector
[18:48:28] Running MetalDisconnector
[18:48:28] Initializing Normalizer
[18:48:28] Running Normalizer
[18:48:28] Running LargestFragmentChooser
[18:48:28] Running Uncharger
[18:48:28] Initializing MetalDisconnector
[18:48:28] Running MetalDisconnector
[18:48:28] Initializing Normalizer
[18:48:28] Running Normalizer
[18:48:28] Initializing MetalDisconnector
[18:48:28] Running MetalDisconnector
[18:48:28] Initializing Normalizer
[18:48:28] Running Normalizer
[18:48:28] Running LargestFragmentChooser
[18:48:28] Running Uncharger
[18:48:28] Initializing MetalDisconnector
[18:48:28] Running MetalDisconnector
[18:48:28] Initializing Normalizer
[18:48:28] Running Normalizer
[18:48:28] Initializing MetalDisconnector
[18:48:28] Running MetalDisconnector
[18:48:28] Initializing Normalizer
[18:48:28] Running Norma

[2026-07-15T12:48:36.293381+00:00] QSAR-04: Reproduced curation counts and molecular identities


In [8]:
# QSAR-05 — Descriptor generation diagnostic and authoritative frozen 217-descriptor matrix
copy_source(INPUTS["descriptors"], DIR["CSV"] / "curated_training_rdkit_descriptors_raw_authoritative.csv", "QSAR-05", description="Hash-verified historical 217-descriptor matrix")

DESCRIPTOR_DIAGNOSTIC = {
    "executed": False,
    "authoritative_downstream_input": "frozen hash-verified 217-descriptor matrix",
    "reason": "Exact historical numerical reproduction requires the frozen matrix; current RDKit versions may change descriptor definitions.",
}
if RUN_DESCRIPTOR_REGENERATION_DIAGNOSTIC:
    X_REGENERATED = descriptor_dataframe(CURATED_FROZEN["smiles_std"])
    write_csv(X_REGENERATED, DIR["Validation"] / "descriptor_matrix_regenerated_current_environment.csv", "QSAR-05", "diagnostic", "Current-RDKit descriptor regeneration")
    assert list(X_REGENERATED.columns) == list(X_FROZEN.columns)
    differences = []
    total_different_cells = 0
    for column in X_FROZEN.columns:
        frozen_values = pd.to_numeric(X_FROZEN[column], errors="coerce").to_numpy(dtype=float)
        regenerated_values = pd.to_numeric(X_REGENERATED[column], errors="coerce").to_numpy(dtype=float)
        nonmatching = ~np.isclose(frozen_values, regenerated_values, rtol=1e-10, atol=1e-10, equal_nan=True)
        n_nonmatching = int(nonmatching.sum())
        if n_nonmatching:
            total_different_cells += n_nonmatching
            differences.append({
                "descriptor": column,
                "n_nonmatching_cells": n_nonmatching,
                "max_absolute_difference": float(np.nanmax(np.abs(frozen_values - regenerated_values))),
            })
    DESCRIPTOR_DIAGNOSTIC.update({
        "executed": True,
        "runtime_rdkit": rdBase.rdkitVersion,
        "shape": list(X_REGENERATED.shape),
        "differing_descriptor_columns": int(len(differences)),
        "total_different_cells": int(total_different_cells),
        "nan_pattern_differences": int(np.sum(np.isnan(X_REGENERATED.to_numpy(dtype=float)) != np.isnan(X_FROZEN.to_numpy(dtype=float)))),
        "interpretation": "The frozen matrix remains authoritative. Differences are reported, never silently replaced.",
    })
    write_csv(pd.DataFrame(differences), DIR["Validation"] / "descriptor_regeneration_drift_by_column.csv", "QSAR-05", "diagnostic", "Descriptor-version drift by column")
write_json(DESCRIPTOR_DIAGNOSTIC, DIR["Validation"] / "descriptor_regeneration_summary.json", "QSAR-05", "diagnostic", "Descriptor regeneration status")

ARCHIVED_SELECTED = pd.read_csv(INPUTS["selected_descriptors"])["selected_descriptor"].astype(str).tolist()
FULL_PREPROCESSOR_REGENERATED = DescriptorPreprocessor(NAN_THRESHOLD, CORR_THRESHOLD).fit(X_FROZEN, Y)
REGENERATED_SELECTED = list(FULL_PREPROCESSOR_REGENERATED.feature_names_)
assert len(ARCHIVED_SELECTED) == 165
assert REGENERATED_SELECTED == ARCHIVED_SELECTED
write_csv(pd.DataFrame({"selected_descriptor": REGENERATED_SELECTED}), DIR["Tables"] / "selected_rdkit_descriptors_regenerated.csv", "QSAR-05", "regenerated", "Selected descriptors reproduced from the frozen matrix")
copy_source(INPUTS["selected_descriptors"], DIR["Supplementary_Output"] / "selected_rdkit_descriptors_archived.csv", "QSAR-05", description="Archived 165-descriptor list")
write_json({
    "raw_descriptors": 217,
    "selected_descriptors": len(REGENERATED_SELECTED),
    "missingness_threshold": NAN_THRESHOLD,
    "imputation": "training-subset median",
    "variance_filter": "zero variance removed",
    "correlation_filter": "absolute Pearson correlation > 0.95; later column dropped",
    "scaling": "StandardScaler fitted only within training folds/subsets",
    "selected_list_matches_archived": REGENERATED_SELECTED == ARCHIVED_SELECTED,
}, DIR["Tables"] / "descriptor_preprocessing_manifest.json", "QSAR-05", description="Descriptor and preprocessing manifest")
log_event("QSAR-05", "Verified authoritative descriptor matrix and 165-feature selection", DESCRIPTOR_DIAGNOSTIC)


[2026-07-15T12:48:58.248038+00:00] QSAR-05: Verified authoritative descriptor matrix and 165-feature selection


In [9]:
# QSAR-06 — Seed-42 random split and fixed model/hyperparameter definitions
INDEX = np.arange(len(CURATED_FROZEN))
INDEX_TRAIN, INDEX_TEST = train_test_split(INDEX, test_size=TEST_SIZE, stratify=Y, random_state=SEED)
X_TRAIN, X_TEST = X_FROZEN.iloc[INDEX_TRAIN], X_FROZEN.iloc[INDEX_TEST]
Y_TRAIN, Y_TEST = Y[INDEX_TRAIN], Y[INDEX_TEST]
SCALE_POS_WEIGHT = float((Y_TRAIN == 0).sum() / (Y_TRAIN == 1).sum())

assert len(INDEX_TRAIN) == 1483 and len(INDEX_TEST) == 371
assert int((Y_TRAIN == 1).sum()) == 1143 and int((Y_TRAIN == 0).sum()) == 340
assert int((Y_TEST == 1).sum()) == 286 and int((Y_TEST == 0).sum()) == 85

split_assignment = CURATED_FROZEN[["molecule_chembl_id", "inchikey_std", "smiles_std", "y", "scaffold"]].copy()
split_assignment["source_row"] = INDEX
split_assignment["random_split"] = ""
split_assignment.loc[INDEX_TRAIN, "random_split"] = "train"
split_assignment.loc[INDEX_TEST, "random_split"] = "test"
write_csv(split_assignment, DIR["CSV"] / "random_seed42_split_assignment.csv", "QSAR-06", "regenerated", "Exact seed-42 stratified split assignment")

MODEL_HYPERPARAMETERS = {
    "Random_Forest": {
        "n_estimators": 500, "max_features": "sqrt", "class_weight": "balanced",
        "random_state": SEED, "n_jobs": N_JOBS,
    },
    "XGBoost": {
        "n_estimators": 500, "max_depth": 5, "learning_rate": 0.03,
        "subsample": 0.9, "colsample_bytree": 0.9,
        "scale_pos_weight": SCALE_POS_WEIGHT, "eval_metric": "logloss",
        "random_state": SEED, "verbosity": 0, "tree_method": "hist", "n_jobs": N_JOBS,
    },
    "SVM_RBF": {
        "kernel": "rbf", "C": 1.0, "gamma": "scale",
        "class_weight": "balanced", "probability": True, "random_state": SEED,
    },
}
write_json({
    "model_parameters": MODEL_HYPERPARAMETERS,
    "hyperparameter_search_status": "No separate classical-QSAR hyperparameter-search artifact was found. The fixed historical parameters are reproduced exactly.",
    "selection_metric": "mean five-fold stratified CV ROC-AUC",
}, DIR["Provenance"] / "classical_qsar_hyperparameter_provenance.json", "QSAR-06", "provenance", "Fixed hyperparameters and search provenance")

MODELS = {
    "Random_Forest": RandomForestClassifier(**MODEL_HYPERPARAMETERS["Random_Forest"]),
    "XGBoost": xgb.XGBClassifier(**MODEL_HYPERPARAMETERS["XGBoost"]),
    "SVM_RBF": SVC(**MODEL_HYPERPARAMETERS["SVM_RBF"]),
}
PIPES = {name: Pipeline([("prep", DescriptorPreprocessor()), ("clf", estimator)]) for name, estimator in MODELS.items()}
CV_SPLITTER = StratifiedKFold(n_splits=N_CV_SPLITS, shuffle=True, random_state=SEED)
SCORING = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "balanced_accuracy": "balanced_accuracy",
    "mcc": make_scorer(matthews_corrcoef),
}
log_event("QSAR-06", "Created exact split and fixed model definitions", {"train": len(INDEX_TRAIN), "test": len(INDEX_TEST), "scale_pos_weight": SCALE_POS_WEIGHT})


[2026-07-15T12:48:58.279310+00:00] QSAR-06: Created exact split and fixed model definitions


In [10]:
# QSAR-07 — Leakage-controlled five-fold model comparison
#
# Scientific policy:
# - The hash-verified archived table is the authoritative manuscript layer.
# - A fresh run is still performed in the full profile, but cross-platform or
#   cross-version differences are exported as diagnostics rather than being
#   mistaken for evidence that the archived result is invalid.
# - Exact assertions are applied to the archived values and checkpoint replays,
#   not to version-sensitive fresh estimator fitting.

CV_ARCHIVED = pd.read_csv(INPUTS["cv_archived"])
CV_REGENERATED = None
CV_FOLD_SCORES = None
CV_COMPARISON = None
CV_EXACT_REPRODUCTION = None

HISTORICAL_CV_TARGETS = {
    "XGBoost": {
        "roc_auc_mean": 0.8886108557419752, "roc_auc_std": 0.021700009293330574,
        "average_precision_mean": 0.9596540533357997, "average_precision_std": 0.007054566044792357,
        "balanced_accuracy_mean": 0.7824009355523409, "balanced_accuracy_std": 0.028000134707361395,
        "mcc_mean": 0.5814270866129284, "mcc_std": 0.04380991322851152,
    },
    "SVM_RBF": {
        "roc_auc_mean": 0.8637292080702654,
        "average_precision_mean": 0.9524232474551132,
        "balanced_accuracy_mean": 0.7844679609375351,
        "mcc_mean": 0.5295461975914943,
    },
}

# Verify the immutable archived source itself against the forensic targets.
for model_name, expected_metrics in HISTORICAL_CV_TARGETS.items():
    archived_row = CV_ARCHIVED.loc[CV_ARCHIVED["model"].eq(model_name)].iloc[0]
    for metric_name, expected_value in expected_metrics.items():
        compare_numeric(archived_row[metric_name], expected_value, 1e-12, f"archived {model_name} {metric_name}")

if EXECUTION_PROFILE == "full":
    summary_rows = []
    fold_rows = []
    for model_name, pipeline in PIPES.items():
        result = cross_validate(
            pipeline, X_TRAIN, Y_TRAIN,
            cv=CV_SPLITTER, scoring=SCORING,
            n_jobs=1, return_train_score=False,
        )
        row = {"model": model_name}
        for metric_name in SCORING:
            values = np.asarray(result[f"test_{metric_name}"], dtype=float)
            row[f"{metric_name}_mean"] = float(values.mean())
            row[f"{metric_name}_std"] = float(values.std())
            for fold_index, value in enumerate(values, start=1):
                fold_rows.append({"model": model_name, "fold": fold_index, "metric": metric_name, "value": float(value)})
        summary_rows.append(row)

    CV_REGENERATED = pd.DataFrame(summary_rows).sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)
    CV_FOLD_SCORES = pd.DataFrame(fold_rows)
    write_csv(CV_REGENERATED, DIR["Tables"] / "model_comparison_cross_validation_current_environment.csv", "QSAR-07", "current_environment_retraining", "Fresh leakage-controlled five-fold model comparison")
    write_csv(CV_FOLD_SCORES, DIR["Validation"] / "model_comparison_fold_scores_current_environment.csv", "QSAR-07", "current_environment_retraining", "Per-fold fresh model metrics")

    CV_COMPARISON = CV_ARCHIVED.merge(
        CV_REGENERATED, on="model", suffixes=("_archived", "_current_environment"), validate="one_to_one"
    )
    metric_columns = [
        "roc_auc_mean", "roc_auc_std", "average_precision_mean", "average_precision_std",
        "balanced_accuracy_mean", "balanced_accuracy_std", "mcc_mean", "mcc_std",
    ]
    for metric in metric_columns:
        CV_COMPARISON[f"abs_diff_{metric}"] = (
            CV_COMPARISON[f"{metric}_archived"] - CV_COMPARISON[f"{metric}_current_environment"]
        ).abs()

    status_rows = []
    for _, row in CV_COMPARISON.iterrows():
        model_name = str(row["model"])
        max_difference = float(max(row[f"abs_diff_{metric}"] for metric in metric_columns))
        exact = bool(max_difference <= FRESH_RETRAINING_MATCH_TOLERANCE)
        status_rows.append({
            "model": model_name,
            "maximum_absolute_metric_difference": max_difference,
            "exact_within_tolerance": exact,
            "tolerance": FRESH_RETRAINING_MATCH_TOLERANCE,
            "interpretation": (
                "Exact current-environment reproduction"
                if exact else
                "Fresh-training drift; archived hash-verified table remains authoritative"
            ),
        })
    CV_REPRODUCIBILITY_STATUS = pd.DataFrame(status_rows)
    exact_models = CV_REPRODUCIBILITY_STATUS.set_index("model")["exact_within_tolerance"].to_dict()
    CV_EXACT_REPRODUCTION = bool(exact_models.get("XGBoost", False) and exact_models.get("SVM_RBF", False))

    write_csv(CV_COMPARISON, DIR["Validation"] / "model_comparison_archived_vs_current_environment.csv", "QSAR-07", "reproducibility_comparison", "Archived-versus-current-environment CV comparison")
    write_csv(CV_REPRODUCIBILITY_STATUS, DIR["Validation"] / "model_comparison_reproducibility_status.csv", "QSAR-07", "reproducibility_comparison", "Per-model exactness classification")
    write_json({
        "exact_xgboost_and_svm_within_tolerance": CV_EXACT_REPRODUCTION,
        "tolerance": FRESH_RETRAINING_MATCH_TOLERANCE,
        "archived_selected_model": "XGBoost",
        "current_environment_selected_model": str(CV_REGENERATED.iloc[0]["model"]),
        "runtime": {
            "python": platform.python_version(),
            "scikit_learn": sklearn.__version__,
            "xgboost": xgb.__version__,
            "operating_system": platform.platform(),
            "n_jobs": N_JOBS,
        },
        "policy": "Archived hash-verified values are used for manuscript outputs whenever fresh retraining is not exact.",
    }, DIR["Validation"] / "fresh_cv_retraining_status.json", "QSAR-07", "reproducibility_comparison", "Fresh CV exactness and environment")

    if not CV_EXACT_REPRODUCTION:
        message = (
            "Fresh CV retraining differs from the archived historical metrics in this environment. "
            "This is recorded as software/platform drift; the notebook will continue using the "
            "hash-verified archived table for manuscript and supplementary outputs."
        )
        warnings.warn(message, RuntimeWarning)
        if STRICT_FRESH_RETRAINING_MATCH:
            raise AssertionError(message)

    for metric in ["roc_auc", "average_precision", "balanced_accuracy", "mcc"]:
        plot_df = CV_REGENERATED.sort_values(f"{metric}_mean")
        fig, ax = plt.subplots(figsize=(7, 4.2))
        ax.barh(plot_df["model"], plot_df[f"{metric}_mean"], xerr=plot_df[f"{metric}_std"], capsize=4)
        ax.set_xlabel(metric.replace("_", " ").title())
        ax.set_title("Current-environment five-fold retraining")
        ax.grid(axis="x", alpha=0.25)
        fig.tight_layout()
        if CV_EXACT_REPRODUCTION:
            figure_path = DIR["Figures"] / f"02_cv_{metric}.png"
            figure_description = f"Exact five-fold CV {metric}"
        else:
            figure_path = DIR["Validation"] / "Diagnostics" / f"02_cv_{metric}_current_environment.png"
            figure_description = f"Current-environment five-fold CV {metric} diagnostic"
        save_figure(fig, figure_path, "QSAR-07", figure_description)
else:
    write_json({
        "status": "fresh_retraining_not_executed_in_checkpoint_replay_profile",
        "authoritative_source": "Hash-verified archived model comparison table",
    }, DIR["Validation"] / "fresh_cv_retraining_status.json", "QSAR-07", "audit_documentation", "Checkpoint-replay CV status")

# Publication outputs always use the immutable historical table. Fresh metrics,
# when available, remain a separate diagnostic layer.
CV_FOR_REPORT = CV_ARCHIVED.copy()
CV_FOR_REPORT["provenance_status"] = "hash_verified_archived_authoritative"
write_csv(CV_FOR_REPORT, DIR["Tables"] / "model_comparison_cross_validation_authoritative.csv", "QSAR-07", "archived_authoritative", "Authoritative historical model comparison table")
log_event("QSAR-07", "Completed model-comparison section", {
    "profile": EXECUTION_PROFILE,
    "authoritative_best_model": "XGBoost",
    "fresh_exact_reproduction": CV_EXACT_REPRODUCTION,
})

C:\Users\Prottoy\AppData\Local\Temp\ipykernel_21260\2419901932.py:116: RuntimeWarning: Fresh CV retraining differs from the archived historical metrics in this environment. This is recorded as software/platform drift; the notebook will continue using the hash-verified archived table for manuscript and supplementary outputs.
  warnings.warn(message, RuntimeWarning)


[2026-07-15T12:49:20.197493+00:00] QSAR-07: Completed model-comparison section


In [11]:
# QSAR-08 — Sigmoid-calibrated XGBoost random held-out validation
#
# The historical aggregate metrics were independently reproduced during the
# forensic audit, but the original per-compound probability vector was not
# preserved. A fresh fit is therefore a reproducibility diagnostic. When its
# metrics drift in a newer environment, the notebook records the drift and keeps
# the audit-verified historical aggregate values as the publication layer.

HISTORICAL_RANDOM_HELDOUT_METRICS = {
    "dataset": "Held-out internal random test set",
    "n": 371, "n_active": 286, "n_inactive": 85,
    "roc_auc": 0.9062937062937062,
    "pr_auc": 0.9665234428396158,
    "balanced_accuracy": 0.7903126285479227,
    "mcc": 0.6106527518357636,
    "f1": 0.9159519725557462,
    "precision": 0.898989898989899,
    "sensitivity": 0.9335664335664335,
    "specificity": 0.6470588235294118,
    "brier_score": 0.0915081198391171,
    "threshold": 0.50,
    "tn": 55, "fp": 30, "fn": 19, "tp": 267,
}
RANDOM_METRICS_FOR_REPORT = dict(HISTORICAL_RANDOM_HELDOUT_METRICS)
RANDOM_METRICS = None
RANDOM_PREDICTIONS = None
RANDOM_CALIBRATED_MODEL = None
RANDOM_EXACT_REPRODUCTION = None
RANDOM_COMPARISON = None

write_csv(
    pd.DataFrame([RANDOM_METRICS_FOR_REPORT]),
    DIR["Tables"] / "random_heldout_validation_metrics_authoritative.csv",
    "QSAR-08", "forensic_audit_verified_historical_target",
    "Authoritative aggregate random held-out metrics independently reproduced during the forensic audit",
)

if EXECUTION_PROFILE == "full":
    RANDOM_CALIBRATED_MODEL = CalibratedClassifierCV(estimator=clone(PIPES["XGBoost"]), method="sigmoid", cv=5)
    RANDOM_CALIBRATED_MODEL.fit(X_TRAIN, Y_TRAIN)
    random_probability = RANDOM_CALIBRATED_MODEL.predict_proba(X_TEST)[:, 1]
    RANDOM_METRICS = metric_row(Y_TEST, random_probability, 0.50, "Held-out internal random test set — current environment")
    RANDOM_PREDICTIONS = CURATED_FROZEN.iloc[INDEX_TEST][["molecule_chembl_id", "smiles_std", "inchikey_std", "y", "activity_label", "scaffold"]].copy()
    RANDOM_PREDICTIONS["source_row"] = INDEX_TEST
    RANDOM_PREDICTIONS["prob_active_current_environment"] = random_probability
    RANDOM_PREDICTIONS["prediction_threshold_0_50_current_environment"] = (random_probability >= 0.50).astype(int)

    metric_keys = [
        "roc_auc", "pr_auc", "balanced_accuracy", "mcc", "f1", "precision",
        "sensitivity", "specificity", "brier_score", "tn", "fp", "fn", "tp",
    ]
    comparison_rows = []
    for key in metric_keys:
        archived_value = float(HISTORICAL_RANDOM_HELDOUT_METRICS[key])
        current_value = float(RANDOM_METRICS[key])
        difference = abs(current_value - archived_value)
        tolerance = 0.0 if key in {"tn", "fp", "fn", "tp"} else FRESH_RETRAINING_MATCH_TOLERANCE
        comparison_rows.append({
            "metric": key,
            "archived_audit_verified": archived_value,
            "current_environment": current_value,
            "absolute_difference": difference,
            "tolerance": tolerance,
            "exact_within_tolerance": bool(difference <= tolerance),
        })
    RANDOM_COMPARISON = pd.DataFrame(comparison_rows)
    RANDOM_EXACT_REPRODUCTION = bool(RANDOM_COMPARISON["exact_within_tolerance"].all())

    write_csv(pd.DataFrame([RANDOM_METRICS]), DIR["Validation"] / "Diagnostics" / "random_heldout_metrics_current_environment.csv", "QSAR-08", "current_environment_retraining", "Fresh calibrated random held-out metrics")
    write_csv(RANDOM_PREDICTIONS, DIR["Validation"] / "Diagnostics" / "random_heldout_predictions_current_environment.csv", "QSAR-08", "current_environment_retraining", "Fresh per-compound calibrated random held-out probabilities")
    write_csv(RANDOM_COMPARISON, DIR["Validation"] / "random_heldout_archived_vs_current_environment.csv", "QSAR-08", "reproducibility_comparison", "Random held-out metric comparison")
    write_json({
        "exact_within_tolerance": RANDOM_EXACT_REPRODUCTION,
        "tolerance": FRESH_RETRAINING_MATCH_TOLERANCE,
        "historical_probability_vector_available": False,
        "policy": "Use audit-verified historical aggregate metrics for publication output when fresh fitting drifts.",
        "runtime": {"scikit_learn": sklearn.__version__, "xgboost": xgb.__version__, "operating_system": platform.platform()},
    }, DIR["Validation"] / "random_heldout_reproducibility_status.json", "QSAR-08", "reproducibility_comparison", "Random held-out exactness status")

    model_path = register_export(DIR["Models"] / "random_heldout_calibrated_xgboost_current_environment.joblib", "QSAR-08", "current_environment_retraining", "Random held-out validation model fitted in this run")
    joblib.dump(RANDOM_CALIBRATED_MODEL, model_path)

    if not RANDOM_EXACT_REPRODUCTION:
        message = (
            "Fresh calibrated random held-out fitting differs from the forensic historical target in this environment. "
            "The current probabilities are retained as diagnostics; authoritative aggregate manuscript values are unchanged."
        )
        warnings.warn(message, RuntimeWarning)
        if STRICT_FRESH_RETRAINING_MATCH:
            raise AssertionError(message)

    figure_prefix = "03_random_heldout" if RANDOM_EXACT_REPRODUCTION else "03_random_heldout_current_environment"
    title_prefix = "Random held-out" if RANDOM_EXACT_REPRODUCTION else "Random held-out — current environment diagnostic"
    plot_roc_pr_calibration(Y_TEST, random_probability, figure_prefix, title_prefix, "QSAR-08")
    plot_confusion(Y_TEST, random_probability, 0.50, f"{title_prefix} confusion matrix", DIR["Figures"] / f"{figure_prefix}_confusion_matrix.png", "QSAR-08")
else:
    write_json({
        "status": "fresh_retraining_not_executed_in_checkpoint_replay_profile",
        "historical_probability_vector_available": False,
        "authoritative_aggregate_metrics": HISTORICAL_RANDOM_HELDOUT_METRICS,
    }, DIR["Validation"] / "random_heldout_reproducibility_status.json", "QSAR-08", "audit_documentation", "Checkpoint-replay random held-out status")

log_event("QSAR-08", "Completed calibrated random held-out section", {
    "profile": EXECUTION_PROFILE,
    "fresh_exact_reproduction": RANDOM_EXACT_REPRODUCTION,
    "publication_layer": "forensic_audit_verified_historical_aggregate_metrics",
})

C:\Users\Prottoy\AppData\Local\Temp\ipykernel_21260\1360994198.py:88: RuntimeWarning: Fresh calibrated random held-out fitting differs from the forensic historical target in this environment. The current probabilities are retained as diagnostics; authoritative aggregate manuscript values are unchanged.
  warnings.warn(message, RuntimeWarning)


[2026-07-15T12:49:28.961420+00:00] QSAR-08: Completed calibrated random held-out section


In [12]:
# QSAR-09 — Y-randomization / label-permutation validation
#
# The complete 30-score historical vector is present and hash-verified. The
# authoritative summary and figure are therefore recomputed directly from that
# vector. Full-profile fresh permutation testing is retained as an environment
# diagnostic and cannot invalidate the preserved historical vector.

Y_ARCHIVED_SCORES = pd.read_csv(INPUTS["y_scores_archived"])["permuted_roc_auc"].to_numpy(dtype=float)
HISTORICAL_OBSERVED_CV_ROC_AUC = 0.8886108557419752
Y_REGENERATED_SCORES = None
Y_REGENERATED_SUMMARY = None
Y_EXACT_REPRODUCTION = None

if EXECUTION_PROFILE == "full":
    observed_regenerated, scores_regenerated, pvalue_regenerated = permutation_test_score(
        clone(PIPES["XGBoost"]), X_TRAIN, Y_TRAIN,
        cv=CV_SPLITTER, scoring="roc_auc", n_permutations=N_PERMUTATIONS,
        n_jobs=N_JOBS, random_state=SEED,
    )
    Y_REGENERATED_SCORES = np.asarray(scores_regenerated, dtype=float)
    Y_REGENERATED_SUMMARY = {
        "observed_cv_roc_auc": float(observed_regenerated),
        "permuted_mean_roc_auc": float(np.mean(Y_REGENERATED_SCORES)),
        "permuted_std_roc_auc": float(np.std(Y_REGENERATED_SCORES)),
        "permutation_pvalue": float(pvalue_regenerated),
        "n_permutations": int(len(Y_REGENERATED_SCORES)),
        "provenance": "current_environment_retraining_diagnostic",
    }
    score_differences = np.abs(Y_REGENERATED_SCORES - Y_ARCHIVED_SCORES)
    Y_EXACT_REPRODUCTION = bool(
        abs(float(observed_regenerated) - HISTORICAL_OBSERVED_CV_ROC_AUC) <= FRESH_RETRAINING_MATCH_TOLERANCE
        and np.max(score_differences) <= FRESH_RETRAINING_MATCH_TOLERANCE
    )
    write_csv(pd.DataFrame({
        "permutation": np.arange(1, len(Y_REGENERATED_SCORES) + 1),
        "archived_permuted_roc_auc": Y_ARCHIVED_SCORES,
        "current_environment_permuted_roc_auc": Y_REGENERATED_SCORES,
        "absolute_difference": score_differences,
    }), DIR["Validation"] / "y_randomization_archived_vs_current_environment.csv", "QSAR-09", "reproducibility_comparison", "Y-randomization archived-versus-fresh comparison")
    write_json({
        **Y_REGENERATED_SUMMARY,
        "exact_historical_reproduction": Y_EXACT_REPRODUCTION,
        "maximum_absolute_permutation_score_difference": float(np.max(score_differences)),
        "tolerance": FRESH_RETRAINING_MATCH_TOLERANCE,
        "runtime": {"scikit_learn": sklearn.__version__, "xgboost": xgb.__version__, "operating_system": platform.platform()},
    }, DIR["Validation"] / "y_randomization_current_environment_status.json", "QSAR-09", "reproducibility_comparison", "Fresh Y-randomization exactness status")
    if not Y_EXACT_REPRODUCTION:
        message = (
            "Fresh Y-randomization differs from the preserved historical vector in this environment. "
            "The hash-verified archived 30-score vector remains the authoritative result."
        )
        warnings.warn(message, RuntimeWarning)
        if STRICT_FRESH_RETRAINING_MATCH:
            raise AssertionError(message)

# Authoritative result: recompute statistics from the hash-verified historical scores.
permutation_scores = Y_ARCHIVED_SCORES.copy()
observed_score = HISTORICAL_OBSERVED_CV_ROC_AUC
permutation_pvalue = (1.0 + float(np.sum(permutation_scores >= observed_score))) / (len(permutation_scores) + 1.0)
Y_RANDOMIZATION_SUMMARY = {
    "observed_cv_roc_auc": float(observed_score),
    "permuted_mean_roc_auc": float(np.mean(permutation_scores)),
    "permuted_std_roc_auc": float(np.std(permutation_scores)),
    "permutation_pvalue": float(permutation_pvalue),
    "n_permutations": int(len(permutation_scores)),
    "provenance": "hash_verified_archived_scores_summary_recomputed",
    "fresh_exact_reproduction": Y_EXACT_REPRODUCTION,
}
compare_numeric(Y_RANDOMIZATION_SUMMARY["observed_cv_roc_auc"], 0.8886108557419752, 1e-12, "Y-randomization observed")
compare_numeric(Y_RANDOMIZATION_SUMMARY["permuted_mean_roc_auc"], 0.504880807394589, 1e-12, "Y-randomization null mean")
compare_numeric(Y_RANDOMIZATION_SUMMARY["permuted_std_roc_auc"], 0.023898815017265835, 1e-12, "Y-randomization null SD")
compare_numeric(Y_RANDOMIZATION_SUMMARY["permutation_pvalue"], 0.03225806451612903, 1e-12, "Y-randomization p-value")

write_csv(pd.DataFrame({"permutation": np.arange(1, len(permutation_scores) + 1), "permuted_roc_auc": permutation_scores}), DIR["Y_Randomization"] / "y_randomization_permutation_scores_authoritative.csv", "QSAR-09", "hash_verified_archived_scores", "Thirty authoritative label-permutation ROC-AUC scores")
write_json(Y_RANDOMIZATION_SUMMARY, DIR["Y_Randomization"] / "y_randomization_summary_authoritative.json", "QSAR-09", "hash_verified_archived_scores_summary_recomputed", "Authoritative Y-randomization summary")
fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.hist(permutation_scores, bins=12, edgecolor="white")
ax.axvline(observed_score, linestyle="--", linewidth=2, label=f"Observed = {observed_score:.4f}")
ax.set(xlabel="Five-fold CV ROC-AUC", ylabel="Permutations", title="QSAR Y-randomization — authoritative historical vector")
ax.legend(); ax.grid(axis="y", alpha=0.2); fig.tight_layout()
save_figure(fig, DIR["Y_Randomization"] / "05_y_randomization_authoritative.png", "QSAR-09", "Y-randomization distribution from hash-verified historical scores")
log_event("QSAR-09", "Validated Y-randomization", Y_RANDOMIZATION_SUMMARY)

[2026-07-15T12:54:04.921860+00:00] QSAR-09: Validated Y-randomization


C:\Users\Prottoy\AppData\Local\Temp\ipykernel_21260\56184081.py:52: RuntimeWarning: Fresh Y-randomization differs from the preserved historical vector in this environment. The hash-verified archived 30-score vector remains the authoritative result.
  warnings.warn(message, RuntimeWarning)


In [13]:
# QSAR-10 — Preserve production checkpoint, regenerate AD quantities, and export portable model bundle
PRODUCTION_MODEL_PATH = copy_source(INPUTS["production_model"], DIR["Models"] / "Historical_Archived" / "AURKB_QSAR_production_calibrated_model.joblib", "QSAR-10", description="Original production calibrated model checkpoint")
AD_PREPROCESSOR_PATH = copy_source(INPUTS["ad_preprocessor"], DIR["Models"] / "Historical_Archived" / "AURKB_QSAR_AD_descriptor_preprocessor.joblib", "QSAR-10", description="Original production AD preprocessor")
AD_MATRIX_PATH = copy_source(INPUTS["ad_matrix"], DIR["Applicability_Domain"] / "AURKB_QSAR_AD_X_full_scaled.npy", "QSAR-10", description="Original scaled full-training reference matrix")
copy_source(INPUTS["artifact_summary"], DIR["Models"] / "Historical_Archived" / "artifact_summary.json", "QSAR-10", description="Original model artifact summary")
copy_source(INPUTS["run_config"], DIR["Provenance"] / "historical_qsar_run_config.json", "QSAR-10", description="Original QSAR run configuration")

PRODUCTION_MODEL, production_model_warnings = load_joblib_with_warnings(INPUTS["production_model"])
AD_PREPROCESSOR, ad_preprocessor_warnings = load_joblib_with_warnings(INPUTS["ad_preprocessor"])
AD_REFERENCE_ARCHIVED = np.load(INPUTS["ad_matrix"])
AD_REFERENCE_REGENERATED = AD_PREPROCESSOR.transform(X_FROZEN)
assert AD_REFERENCE_ARCHIVED.shape == (1854, 165)
AD_REFERENCE_MAX_ABS_DIFF = float(np.max(np.abs(AD_REFERENCE_ARCHIVED - AD_REFERENCE_REGENERATED)))
AD_REFERENCE_NUMERICAL_TOLERANCE = 1e-12
assert AD_REFERENCE_MAX_ABS_DIFF <= AD_REFERENCE_NUMERICAL_TOLERANCE
PRODUCTION_H_STAR = h_star(AD_REFERENCE_ARCHIVED.shape[0], AD_REFERENCE_ARCHIVED.shape[1])
compare_numeric(PRODUCTION_H_STAR, 0.2686084142394822, 1e-15, "production h-star")

production_training_probability = PRODUCTION_MODEL.predict_proba(X_FROZEN)[:, 1]
training_checkpoint_output = CURATED_FROZEN[["molecule_chembl_id", "inchikey_std", "smiles_std", "y"]].copy()
training_checkpoint_output["archived_production_probability"] = production_training_probability
training_checkpoint_output["archived_production_prediction_0_50"] = (production_training_probability >= 0.50).astype(int)
write_csv(training_checkpoint_output, DIR["CSV"] / "production_checkpoint_training_set_probabilities.csv", "QSAR-10", "checkpoint_replay", "Probabilities regenerated by the archived production checkpoint")

PORTABLE_PRODUCTION_DIR = DIR["Models"] / "Portable_Production_Bundle"
portable_fold_records = []
portable_fold_probabilities = []
for fold_index, calibrated_classifier in enumerate(PRODUCTION_MODEL.calibrated_classifiers_, start=1):
    pipeline = calibrated_classifier.estimator
    preprocessor = pipeline.named_steps["prep"]
    classifier = pipeline.named_steps["clf"]
    calibrator = calibrated_classifier.calibrators[0]
    metadata, arrays, metadata_path, arrays_path = serialize_preprocessor(preprocessor, PORTABLE_PRODUCTION_DIR, f"fold_{fold_index}")
    booster_path = PORTABLE_PRODUCTION_DIR / f"fold_{fold_index}_xgboost.json"
    classifier.get_booster().save_model(str(booster_path))
    calibration_path = PORTABLE_PRODUCTION_DIR / f"fold_{fold_index}_sigmoid_calibration.json"
    calibration_path.write_text(json.dumps({"a": float(calibrator.a_), "b": float(calibrator.b_), "formula": "1/(1+exp(a*raw_probability+b))"}, indent=2), encoding="utf-8")
    for generated_path, description in [
        (metadata_path, f"Portable fold {fold_index} preprocessor metadata"),
        (arrays_path, f"Portable fold {fold_index} scaler arrays"),
        (booster_path, f"Portable fold {fold_index} XGBoost model"),
        (calibration_path, f"Portable fold {fold_index} sigmoid calibration"),
    ]:
        register_export(generated_path, "QSAR-10", "portable_checkpoint", description)
    Z = portable_transform(X_FROZEN, metadata, arrays)
    booster = xgb.Booster()
    booster.load_model(str(booster_path))
    raw_probability = booster.predict(xgb.DMatrix(Z))
    calibrated_probability = sigmoid_calibration(raw_probability, float(calibrator.a_), float(calibrator.b_))
    portable_fold_probabilities.append(calibrated_probability)
    portable_fold_records.append({
        "fold": fold_index,
        "n_features": len(metadata["feature_names"]),
        "calibration_a": float(calibrator.a_),
        "calibration_b": float(calibrator.b_),
        "booster_sha256": sha256_file(booster_path),
    })
portable_probability = np.mean(np.vstack(portable_fold_probabilities), axis=0)
PORTABLE_PRODUCTION_MAX_DIFF = float(np.max(np.abs(portable_probability - production_training_probability)))
PORTABLE_PRODUCTION_TOLERANCE = 1e-12
assert PORTABLE_PRODUCTION_MAX_DIFF <= PORTABLE_PRODUCTION_TOLERANCE
write_csv(pd.DataFrame(portable_fold_records), PORTABLE_PRODUCTION_DIR / "portable_production_bundle_index.csv", "QSAR-10", "portable_checkpoint", "Portable production bundle fold index")
write_json({
    "source_checkpoint_sha256": sha256_file(INPUTS["production_model"]),
    "n_calibrated_folds": len(portable_fold_records),
    "portable_vs_joblib_max_abs_probability_difference": PORTABLE_PRODUCTION_MAX_DIFF,
    "numerical_equivalence_tolerance": PORTABLE_PRODUCTION_TOLERANCE,
    "numerically_equivalent_within_tolerance": True,
    "historical_sklearn_serialization_version": "1.6.1",
    "load_warnings": production_model_warnings,
    "purpose": "Future inference without relying on the custom-class joblib pickle alone.",
}, PORTABLE_PRODUCTION_DIR / "portable_production_bundle_manifest.json", "QSAR-10", "portable_checkpoint", "Portable production model manifest")

write_json({
    "n_training_compounds": int(AD_REFERENCE_ARCHIVED.shape[0]),
    "n_selected_descriptors": int(AD_REFERENCE_ARCHIVED.shape[1]),
    "h_star": float(PRODUCTION_H_STAR),
    "archived_matrix_reproduced_within_floating_tolerance": True,
    "archived_matrix_max_abs_difference": AD_REFERENCE_MAX_ABS_DIFF,
    "archived_matrix_numerical_tolerance": AD_REFERENCE_NUMERICAL_TOLERANCE,
    "interpretation": "Differences below tolerance are floating-point roundoff from cross-version deserialization, not descriptor or preprocessing drift.",
    "production_model_load_warnings": production_model_warnings,
    "ad_preprocessor_load_warnings": ad_preprocessor_warnings,
}, DIR["Applicability_Domain"] / "production_applicability_domain_manifest.json", "QSAR-10", "checkpoint_replay", "Production AD reconstruction and checkpoint status")
log_event("QSAR-10", "Preserved production checkpoint and generated numerically equivalent portable bundle", {"portable_max_diff": PORTABLE_PRODUCTION_MAX_DIFF, "tolerance": PORTABLE_PRODUCTION_TOLERANCE, "h_star": PRODUCTION_H_STAR})


[2026-07-15T12:54:06.367139+00:00] QSAR-10: Preserved production checkpoint and generated numerically equivalent portable bundle


In [14]:
# QSAR-11 — Full-library screening reassessment and exact archived QSAR-to-GCN transfer preservation
PRESCREEN = pd.read_csv(INPUTS["prescreen"], low_memory=False)
CANDIDATE_SNAPSHOT = pd.read_csv(INPUTS["candidate_snapshot"], low_memory=False)
assert len(PRESCREEN) == 86056
assert len(CANDIDATE_SNAPSHOT) == 6232
assert CANDIDATE_SNAPSHOT["identifier"].duplicated().sum() == 0
assert bool(CANDIDATE_SNAPSHOT["inside_AD"].astype(bool).all())
assert bool(CANDIDATE_SNAPSHOT["high_confidence_qsar_inside_AD"].astype(bool).all())
assert bool((CANDIDATE_SNAPSHOT["prob_active_qsar"] >= HIGH_CONFIDENCE_THRESHOLD).all())

copy_source(INPUTS["candidate_snapshot"], DIR["CSV"] / "Archived_Historical_Screening" / "AURKB_QSAR_to_GCN_exact_6232_candidate_snapshot.csv", "QSAR-11", description="Exact historical 6,232-row QSAR-to-GCN transfer snapshot")

with zipfile.ZipFile(LOCKED_ZIP) as zf:
    locked_names = zf.namelist()
search_terms = ["all_screened", "inside_AD_all", "high_confidence_inside_AD", "screening_summary"]
archive_search_rows = []
for term in search_terms:
    matches = [name for name in locked_names if term.lower() in name.lower()]
    archive_search_rows.append({"search_term": term, "n_matches": len(matches), "matches": " | ".join(matches[:20])})
write_csv(pd.DataFrame(archive_search_rows), DIR["Provenance"] / "full_screening_artifact_archive_search.csv", "QSAR-11", "provenance", "Archive-wide search for missing original full-screening files")

SCREENING_REPRODUCIBILITY_STATUS = {
    "prescreened_input_rows_verified": 86056,
    "invalid_smiles_reported": 0,
    "reported_inside_descriptor_AD": 34721,
    "reported_predicted_active_probability_ge_0_50": 31020,
    "reported_high_confidence_inside_AD_probability_ge_0_75": 6232,
    "exact_6232_row_transfer_snapshot_preserved": True,
    "upstream_full_screening_exactly_regenerated": False,
    "missing_historical_artifacts": [
        "Original 86,056-row all-screened QSAR CSV containing every probability and leverage value",
        "Exact QSAR software environment, including the unrecorded XGBoost training version",
        "Direct QSAR-run RDKit version record; same-day prescreen evidence indicates RDKit 2026.03.2 but this remains an inference",
    ],
    "forensic_audit_current_environment_replay": {
        "inside_AD": 29715,
        "predicted_active_probability_ge_0_50": 31018,
        "high_confidence_inside_AD_probability_ge_0_75": 4587,
        "candidate_overlap_with_archived_6232": 4586,
    },
    "scientific_decision": "Preserve the verified 6,232-row transfer snapshot unchanged. Do not fabricate or relabel a current-environment rescreen as the historical manuscript output.",
}
write_json(SCREENING_REPRODUCIBILITY_STATUS, DIR["Validation"] / "full_library_screening_reproducibility_status.json", "QSAR-11", "audit_documentation", "Authoritative screening reproducibility conclusion")
write_json({
    "n_candidates": int(len(CANDIDATE_SNAPSHOT)),
    "minimum_qsar_probability": float(CANDIDATE_SNAPSHOT["prob_active_qsar"].min()),
    "maximum_qsar_probability": float(CANDIDATE_SNAPSHOT["prob_active_qsar"].max()),
    "maximum_leverage": float(CANDIDATE_SNAPSHOT["leverage"].max()),
    "all_inside_AD": bool(CANDIDATE_SNAPSHOT["inside_AD"].astype(bool).all()),
    "all_high_confidence": bool(CANDIDATE_SNAPSHOT["high_confidence_qsar_inside_AD"].astype(bool).all()),
}, DIR["Tables"] / "archived_QSAR_to_GCN_snapshot_audit.json", "QSAR-11", "checkpoint_replay", "Independent row-level audit of the 6,232 snapshot")

if RUN_NONAUTHORITATIVE_FULL_SCREENING_DIAGNOSTIC:
    diagnostic_dir = DIR["Validation"] / "NonAuthoritative_Full_Screening_Replay"
    descriptor_names = [item[0] for item in Descriptors.descList]
    calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)
    inverse = np.linalg.pinv(AD_REFERENCE_ARCHIVED.T @ AD_REFERENCE_ARCHIVED)
    diagnostic_parts = []
    invalid_parts = []
    for start in range(0, len(PRESCREEN), 5000):
        chunk = PRESCREEN.iloc[start:start + 5000].copy()
        standardized = chunk["smiles"].apply(standardize_smiles)
        valid = standardized.notna()
        invalid_parts.append(chunk.loc[~valid].copy())
        valid_chunk = chunk.loc[valid].copy()
        smiles_valid = standardized.loc[valid]
        descriptor_rows = []
        for smiles in smiles_valid:
            molecule = Chem.MolFromSmiles(str(smiles))
            descriptor_rows.append(calculator.CalcDescriptors(molecule) if molecule is not None else [np.nan] * len(descriptor_names))
        D = pd.DataFrame(descriptor_rows, columns=descriptor_names).replace([np.inf, -np.inf], np.nan)
        probability = PRODUCTION_MODEL.predict_proba(D)[:, 1]
        Z = AD_PREPROCESSOR.transform(D)
        leverage = np.einsum("ij,jk,ik->i", Z, inverse, Z, optimize=True)
        diagnostic_parts.append(pd.DataFrame({
            "identifier": valid_chunk["identifier"].to_numpy(),
            "smiles": smiles_valid.to_numpy(),
            "prob_active_qsar": probability,
            "prediction_qsar": (probability >= QSAR_ACTIVE_THRESHOLD).astype(int),
            "leverage": leverage,
            "inside_AD": leverage < PRODUCTION_H_STAR,
            "high_confidence_qsar_inside_AD": (probability >= HIGH_CONFIDENCE_THRESHOLD) & (leverage < PRODUCTION_H_STAR),
        }))
    diagnostic_screened = pd.concat(diagnostic_parts, ignore_index=True)
    write_csv(diagnostic_screened, diagnostic_dir / "NONAUTHORITATIVE_current_environment_all_screened.csv", "QSAR-11", "diagnostic", "Current-environment full-screening diagnostic; never manuscript output")
    diagnostic_summary = {
        "valid": int(len(diagnostic_screened)),
        "invalid": int(sum(len(part) for part in invalid_parts)),
        "inside_AD": int(diagnostic_screened["inside_AD"].sum()),
        "predicted_active_0_50": int(diagnostic_screened["prediction_qsar"].sum()),
        "high_confidence_inside_AD_0_75": int(diagnostic_screened["high_confidence_qsar_inside_AD"].sum()),
        "status": "NONAUTHORITATIVE_DIAGNOSTIC",
    }
    write_json(diagnostic_summary, diagnostic_dir / "NONAUTHORITATIVE_current_environment_screening_summary.json", "QSAR-11", "diagnostic", "Non-authoritative screening diagnostic summary")
log_event("QSAR-11", "Preserved exact QSAR-to-GCN snapshot and documented unrecoverable upstream screening", SCREENING_REPRODUCIBILITY_STATUS)


[2026-07-15T12:54:07.015033+00:00] QSAR-11: Preserved exact QSAR-to-GCN snapshot and documented unrecoverable upstream screening


In [15]:
# QSAR-11A — Automated QSAR-to-GCN transfer confirmation
# Read-only audit: compares the archive snapshot with the isolated QSAR-11 copy.
# It does not modify models, predictions, archives, or scientific outputs.

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

EXPECTED_ARCHIVED_SNAPSHOT_SHA256 = (
    "8821947e92d8b209d291b627bae3b1d5068c6e2f1c0154f695abf0e73673f4df"
)

ARCHIVED_SNAPSHOT_KEY = "candidate_snapshot"


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)

    return digest.hexdigest()


def sha256_text(values):
    payload = "\n".join(
        "" if pd.isna(value) else str(value)
        for value in values
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def canonical_bool(series):
    normalized = series.astype(str).str.strip().str.lower()

    valid_values = {
        "true", "false",
        "1", "0",
        "yes", "no",
        "y", "n",
        "t", "f",
    }

    unknown = sorted(set(normalized.dropna()) - valid_values)

    if unknown:
        raise ValueError(
            f"Unrecognized Boolean values: {unknown[:20]}"
        )

    return normalized.isin(["true", "1", "yes", "y", "t"])


def find_column(frame, candidates, label):
    lookup = {
        str(column).strip().lower(): column
        for column in frame.columns
    }

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    raise ValueError(
        f"{label} has no accepted column. "
        f"Expected one of {candidates}; found {list(frame.columns)}"
    )


def prepare_transfer(frame, label):
    frame = frame.copy()

    identifier_col = find_column(frame, ["identifier"], label)
    smiles_col = find_column(frame, ["smiles"], label)

    probability_col = find_column(
        frame,
        ["prob_active_qsar", "probactiveqsar"],
        label,
    )

    inside_ad_col = find_column(
        frame,
        ["inside_AD", "insideAD"],
        label,
    )

    output = pd.DataFrame(
        {
            "identifier": (
                frame[identifier_col].astype(str).str.strip()
            ),
            "smiles": (
                frame[smiles_col].astype(str).str.strip()
            ),
            "prob_active_qsar": (
                pd.to_numeric(
                    frame[probability_col],
                    errors="raise",
                ).astype("float64")
            ),
            "inside_AD": canonical_bool(frame[inside_ad_col]),
        }
    )

    if output["identifier"].duplicated().any():
        examples = output.loc[
            output["identifier"].duplicated(),
            "identifier",
        ].head(20).tolist()

        raise ValueError(
            f"{label} contains duplicate identifiers: {examples}"
        )

    if output["smiles"].eq("").any():
        raise ValueError(f"{label} contains blank SMILES.")

    return output.reset_index(drop=True)


# Locate archive source and QSAR-11 isolated transfer copy.
if ARCHIVED_SNAPSHOT_KEY not in INPUTS:
    raise KeyError(
        f"INPUTS lacks '{ARCHIVED_SNAPSHOT_KEY}'. "
        f"Available keys: {sorted(INPUTS.keys())}"
    )

archived_path = Path(INPUTS[ARCHIVED_SNAPSHOT_KEY]).resolve()

transfer_path = (
    DIR["CSV"]
    / "Archived_Historical_Screening"
    / "AURKB_QSAR_to_GCN_exact_6232_candidate_snapshot.csv"
)

if not archived_path.is_file():
    raise FileNotFoundError(
        f"Archived candidate snapshot is missing:\n{archived_path}"
    )

if not transfer_path.is_file():
    raise FileNotFoundError(
        "The QSAR-11 isolated transfer copy is missing:\n"
        f"{transfer_path}\n\n"
        "Run QSAR-11 before running this audit cell."
    )

print("Archived source snapshot:")
print(archived_path)

print("\nQSAR-11 isolated transfer copy:")
print(transfer_path)


# Load and standardize only the GCN-transfer fields.
archived_raw = pd.read_csv(archived_path, low_memory=False)
transfer_raw = pd.read_csv(transfer_path, low_memory=False)

archived = prepare_transfer(
    archived_raw,
    "Archived GCN candidate snapshot",
)

transfer = prepare_transfer(
    transfer_raw,
    "QSAR-11 isolated transfer copy",
)


# Exact ordered comparison.
same_row_count = len(archived) == len(transfer)

same_ordered_transfer_fields = (
    same_row_count
    and archived.equals(transfer)
)

maximum_probability_absolute_difference = (
    float(
        np.max(
            np.abs(
                archived["prob_active_qsar"]
                - transfer["prob_active_qsar"]
            )
        )
    )
    if same_row_count
    else None
)


# Identifier-level diagnostic comparison.
comparison = archived.merge(
    transfer,
    how="outer",
    on="identifier",
    suffixes=("_archived", "_transfer"),
    indicator=True,
)

comparison["smiles_match"] = comparison[
    "smiles_archived"
].eq(comparison["smiles_transfer"])

comparison["probability_match"] = np.isclose(
    comparison["prob_active_qsar_archived"],
    comparison["prob_active_qsar_transfer"],
    rtol=0.0,
    atol=0.0,
    equal_nan=True,
)

comparison["inside_ad_match"] = comparison[
    "inside_AD_archived"
].eq(comparison["inside_AD_transfer"])

mismatches = comparison.loc[
    comparison["_merge"].ne("both")
    | ~comparison["smiles_match"]
    | ~comparison["probability_match"]
    | ~comparison["inside_ad_match"]
].copy()


# Final report.
archived_sha256 = sha256_file(archived_path)
transfer_sha256 = sha256_file(transfer_path)

report = {
    "audit_name": (
        "QSAR-to-GCN exact historical transfer confirmation"
    ),
    "archived_source": str(archived_path),
    "isolated_transfer_copy": str(transfer_path),
    "expected_archived_sha256": (
        EXPECTED_ARCHIVED_SNAPSHOT_SHA256
    ),
    "archived_sha256": archived_sha256,
    "transfer_copy_sha256": transfer_sha256,
    "archived_sha256_verified": (
        archived_sha256
        == EXPECTED_ARCHIVED_SNAPSHOT_SHA256
    ),
    "archived_rows": int(len(archived)),
    "transfer_rows": int(len(transfer)),
    "same_row_count": bool(same_row_count),
    "same_ordered_identifier_smiles_probability_insideAD": bool(
        same_ordered_transfer_fields
    ),
    "ordered_identifier_hash_archived": sha256_text(
        archived["identifier"]
    ),
    "ordered_identifier_hash_transfer": sha256_text(
        transfer["identifier"]
    ),
    "ordered_smiles_hash_archived": sha256_text(
        archived["smiles"]
    ),
    "ordered_smiles_hash_transfer": sha256_text(
        transfer["smiles"]
    ),
    "ordered_probability_hash_archived": sha256_text(
        archived["prob_active_qsar"].map(
            lambda value: f"{value:.17g}"
        )
    ),
    "ordered_probability_hash_transfer": sha256_text(
        transfer["prob_active_qsar"].map(
            lambda value: f"{value:.17g}"
        )
    ),
    "ordered_inside_ad_hash_archived": sha256_text(
        archived["inside_AD"].astype(str)
    ),
    "ordered_inside_ad_hash_transfer": sha256_text(
        transfer["inside_AD"].astype(str)
    ),
    "maximum_probability_absolute_difference": (
        maximum_probability_absolute_difference
    ),
    "identifier_level_mismatch_count": int(len(mismatches)),
}

report["overall_status"] = (
    "PASS"
    if (
        report["archived_sha256_verified"]
        and report["same_row_count"]
        and report[
            "same_ordered_identifier_smiles_probability_insideAD"
        ]
        and report["identifier_level_mismatch_count"] == 0
    )
    else "FAIL"
)


# Save outputs only inside the current unique run folder.
write_json(
    report,
    DIR["Provenance"]
    / "qsar_to_gcn_exact_transfer_confirmation.json",
    "QSAR-11A",
    "provenance",
    "Automated verification of archived QSAR-to-GCN transfer copy",
)

write_csv(
    pd.DataFrame([report]),
    DIR["Tables"]
    / "qsar_to_gcn_exact_transfer_confirmation.csv",
    "QSAR-11A",
    "provenance",
    "One-row QSAR-to-GCN transfer confirmation report",
)

write_csv(
    mismatches,
    DIR["Tables"]
    / "qsar_to_gcn_exact_transfer_mismatches.csv",
    "QSAR-11A",
    "provenance",
    "Identifier-level mismatch table; empty when confirmation passes",
)


print("\n" + "=" * 72)
print(f"QSAR -> GCN TRANSFER CONFIRMATION: {report['overall_status']}")
print("=" * 72)
print(json.dumps(report, indent=2))

if report["overall_status"] != "PASS":
    print("\nFirst mismatches:")
    display(mismatches.head(20))

    raise AssertionError(
        "QSAR-to-GCN transfer confirmation FAILED. "
        "Inspect qsar_to_gcn_exact_transfer_mismatches.csv."
    )

print(
    "\nPASS: QSAR-11 isolated the exact archived 6,232-row "
    "GCN transfer snapshot in this unique run folder."
)

Archived source snapshot:
C:\Users\Prottoy\Downloads\Notebook_2_Retrail\Inputs\QSAR_Final_Reproduction\run_20260715T124827Z_79cebaa2\Provenance\Input_Cache\locked\gcn_candidate_input_audit.csv

QSAR-11 isolated transfer copy:
C:\Users\Prottoy\Downloads\Notebook_2_Retrail\Inputs\QSAR_Final_Reproduction\run_20260715T124827Z_79cebaa2\CSV\Archived_Historical_Screening\AURKB_QSAR_to_GCN_exact_6232_candidate_snapshot.csv

QSAR -> GCN TRANSFER CONFIRMATION: PASS
{
  "audit_name": "QSAR-to-GCN exact historical transfer confirmation",
  "archived_source": "C:\\Users\\Prottoy\\Downloads\\Notebook_2_Retrail\\Inputs\\QSAR_Final_Reproduction\\run_20260715T124827Z_79cebaa2\\Provenance\\Input_Cache\\locked\\gcn_candidate_input_audit.csv",
  "isolated_transfer_copy": "C:\\Users\\Prottoy\\Downloads\\Notebook_2_Retrail\\Inputs\\QSAR_Final_Reproduction\\run_20260715T124827Z_79cebaa2\\CSV\\Archived_Historical_Screening\\AURKB_QSAR_to_GCN_exact_6232_candidate_snapshot.csv",
  "expected_archived_sha256": "8

In [16]:
# QSAR-12 — Corrected scaffold-disjoint validation: zero-overlap split and exact checkpoint replay
SCAFFOLD_ASSIGNMENT = pd.read_csv(INPUTS["scaffold_assignment"])
SCAFFOLD_REFERENCE_PREDICTIONS = pd.read_csv(INPUTS["scaffold_predictions"])
SCAFFOLD_ARCHIVED_METRICS = pd.read_csv(INPUTS["scaffold_metrics"])

alignment = CURATED_FROZEN.reset_index(names="source_row").merge(
    SCAFFOLD_ASSIGNMENT[["inchikey_std", "scaffold_validation_split"]],
    on="inchikey_std", how="left", validate="one_to_one",
)
if alignment["scaffold_validation_split"].isna().any():
    raise ValueError("Scaffold split assignment failed to align to every curated compound.")

split_rows = {}
for split_name in ["train", "calibration", "test"]:
    split_rows[split_name] = alignment.loc[alignment["scaffold_validation_split"].eq(split_name), "source_row"].to_numpy(dtype=int)

split_summary_regenerated = []
for split_name, rows in split_rows.items():
    subset = CURATED_FROZEN.iloc[rows]
    split_summary_regenerated.append({
        "split": split_name,
        "n_compounds": int(len(subset)),
        "n_scaffolds": int(subset["scaffold"].nunique()),
        "n_active": int(subset["y"].sum()),
        "n_inactive": int((1 - subset["y"]).sum()),
        "active_fraction": float(subset["y"].mean()),
    })
SCAFFOLD_SPLIT_SUMMARY_REGENERATED = pd.DataFrame(split_summary_regenerated)
expected_counts = {"train": (1062, 516), "calibration": (348, 173), "test": (444, 173)}
for split_name, (expected_n, expected_scaffolds) in expected_counts.items():
    row = SCAFFOLD_SPLIT_SUMMARY_REGENERATED.loc[SCAFFOLD_SPLIT_SUMMARY_REGENERATED["split"].eq(split_name)].iloc[0]
    assert int(row["n_compounds"]) == expected_n and int(row["n_scaffolds"]) == expected_scaffolds

scaffold_sets = {name: set(CURATED_FROZEN.iloc[rows]["scaffold"]) for name, rows in split_rows.items()}
overlap_regenerated = {
    "train_calibration_scaffold_overlap": len(scaffold_sets["train"] & scaffold_sets["calibration"]),
    "train_test_scaffold_overlap": len(scaffold_sets["train"] & scaffold_sets["test"]),
    "calibration_test_scaffold_overlap": len(scaffold_sets["calibration"] & scaffold_sets["test"]),
}
assert overlap_regenerated == {
    "train_calibration_scaffold_overlap": 0,
    "train_test_scaffold_overlap": 0,
    "calibration_test_scaffold_overlap": 0,
}
write_csv(SCAFFOLD_SPLIT_SUMMARY_REGENERATED, DIR["Tables"] / "scaffold_split_summary_regenerated.csv", "QSAR-12", "regenerated", "Scaffold train/calibration/test summary")
write_json(overlap_regenerated, DIR["Validation"] / "scaffold_split_overlap_regenerated.json", "QSAR-12", "regenerated", "Independent zero-overlap scaffold audit")
write_csv(SCAFFOLD_ASSIGNMENT, DIR["CSV"] / "scaffold_compound_split_assignment.csv", "QSAR-12", "archived_copy", "Exact historical scaffold assignment")

SCAFFOLD_PIPELINE, scaffold_pipeline_warnings = load_joblib_with_warnings(INPUTS["scaffold_pipeline"])
SCAFFOLD_CALIBRATOR, scaffold_calibrator_warnings = load_joblib_with_warnings(INPUTS["scaffold_calibrator"])
train_rows = split_rows["train"]
test_rows = split_rows["test"]
raw_scaffold_probability = SCAFFOLD_PIPELINE.predict_proba(X_FROZEN.iloc[test_rows])[:, 1]
calibrated_scaffold_probability = SCAFFOLD_CALIBRATOR.predict_proba(safe_logit(raw_scaffold_probability))[:, 1]
Y_SCAFFOLD_TEST = Y[test_rows]

scaffold_preprocessor = SCAFFOLD_PIPELINE.named_steps["prep"]
Z_SCAFFOLD_TRAIN = scaffold_preprocessor.transform(X_FROZEN.iloc[train_rows])
Z_SCAFFOLD_TEST = scaffold_preprocessor.transform(X_FROZEN.iloc[test_rows])
SCAFFOLD_LEVERAGE = calculate_leverage(Z_SCAFFOLD_TRAIN, Z_SCAFFOLD_TEST)
SCAFFOLD_H_STAR = h_star(len(train_rows), Z_SCAFFOLD_TRAIN.shape[1])
SCAFFOLD_INSIDE_AD = SCAFFOLD_LEVERAGE <= SCAFFOLD_H_STAR
compare_numeric(SCAFFOLD_H_STAR, 0.4576271186440678, 1e-15, "scaffold h-star")
assert Z_SCAFFOLD_TRAIN.shape[1] == 161
assert int(SCAFFOLD_INSIDE_AD.sum()) == 420

SCAFFOLD_METRICS = metric_row(Y_SCAFFOLD_TEST, calibrated_scaffold_probability, 0.50, "Corrected scaffold-disjoint QSAR test set")
SCAFFOLD_INSIDE_METRICS = metric_row(Y_SCAFFOLD_TEST[SCAFFOLD_INSIDE_AD], calibrated_scaffold_probability[SCAFFOLD_INSIDE_AD], 0.50, "Corrected scaffold-disjoint QSAR test set inside training AD")

expected_scaffold = {
    "roc_auc": 0.8530772294213155, "pr_auc": 0.9541811834628455,
    "balanced_accuracy": 0.6797628894403088, "mcc": 0.42794093472968275,
    "brier_score": 0.11604172237917644,
    "tn": 39, "fp": 54, "fn": 21, "tp": 330,
}
# The authoritative inside-AD Brier score below is recomputed directly from the
# 420 archived per-compound probabilities and inside-AD flags. It is also the value
# stored in scaffold_qsar_metrics.csv; an earlier audit narrative transcription
# (0.1121292196662012) was incorrect and is not propagated here.
expected_inside = {
    "roc_auc": 0.8551941609977325, "pr_auc": 0.9568648336660653,
    "balanced_accuracy": 0.6845238095238095, "mcc": 0.43425715637740747,
    "brier_score": 0.11205138245368548,
    "tn": 36, "fp": 48, "fn": 20, "tp": 316,
}
for key, value in expected_scaffold.items(): compare_numeric(SCAFFOLD_METRICS[key], value, 0 if key in {"tn", "fp", "fn", "tp"} else 1e-12, f"scaffold {key}")
for key, value in expected_inside.items(): compare_numeric(SCAFFOLD_INSIDE_METRICS[key], value, 0 if key in {"tn", "fp", "fn", "tp"} else 1e-12, f"scaffold inside-AD {key}")

scaffold_predictions = CURATED_FROZEN.iloc[test_rows][["molecule_chembl_id", "smiles_std", "inchikey_std", "activity_nM_median", "pActivity_median", "y", "activity_label", "scaffold"]].copy()
scaffold_predictions["raw_qsar_probability"] = raw_scaffold_probability
scaffold_predictions["calibrated_qsar_probability"] = calibrated_scaffold_probability
scaffold_predictions["predicted_active_threshold_0_50"] = (calibrated_scaffold_probability >= 0.50).astype(int)
scaffold_predictions["scaffold_validation_leverage"] = SCAFFOLD_LEVERAGE
scaffold_predictions["scaffold_validation_inside_train_AD"] = SCAFFOLD_INSIDE_AD
write_csv(pd.DataFrame([SCAFFOLD_METRICS, SCAFFOLD_INSIDE_METRICS]), DIR["Tables"] / "scaffold_qsar_metrics_regenerated.csv", "QSAR-12", "checkpoint_replay", "Exact corrected scaffold metrics")
write_csv(scaffold_predictions, DIR["CSV"] / "scaffold_qsar_predictions_regenerated.csv", "QSAR-12", "checkpoint_replay", "Exact corrected scaffold per-compound probabilities and leverage")

prediction_comparison = SCAFFOLD_REFERENCE_PREDICTIONS[["inchikey_std", "raw_qsar_probability", "calibrated_qsar_probability", "scaffold_validation_leverage"]].merge(
    scaffold_predictions[["inchikey_std", "raw_qsar_probability", "calibrated_qsar_probability", "scaffold_validation_leverage"]],
    on="inchikey_std", suffixes=("_archived", "_replayed"), validate="one_to_one",
)
SCAFFOLD_REPLAY_COMPARISON = {
    "n": int(len(prediction_comparison)),
    "max_abs_raw_probability_difference": float(np.max(np.abs(prediction_comparison["raw_qsar_probability_archived"] - prediction_comparison["raw_qsar_probability_replayed"]))),
    "max_abs_calibrated_probability_difference": float(np.max(np.abs(prediction_comparison["calibrated_qsar_probability_archived"] - prediction_comparison["calibrated_qsar_probability_replayed"]))),
    "max_abs_leverage_difference": float(np.max(np.abs(prediction_comparison["scaffold_validation_leverage_archived"] - prediction_comparison["scaffold_validation_leverage_replayed"]))),
    "pipeline_load_warnings": scaffold_pipeline_warnings,
    "calibrator_load_warnings": scaffold_calibrator_warnings,
}
write_json(SCAFFOLD_REPLAY_COMPARISON, DIR["Validation"] / "scaffold_checkpoint_replay_comparison.json", "QSAR-12", "checkpoint_replay", "Archived-versus-replayed scaffold predictions")

copy_source(INPUTS["scaffold_pipeline"], DIR["Models"] / "Historical_Archived" / "scaffold_validation_train_only_qsar_pipeline.joblib", "QSAR-12", description="Historical train-only scaffold model")
copy_source(INPUTS["scaffold_calibrator"], DIR["Models"] / "Historical_Archived" / "scaffold_validation_sigmoid_calibrator.joblib", "QSAR-12", description="Historical external scaffold sigmoid calibrator")
copy_source(INPUTS["scaffold_config"], DIR["Provenance"] / "historical_scaffold_validation_config.json", "QSAR-12", description="Historical scaffold validation configuration")
copy_source(INPUTS["scaffold_readme"], DIR["Provenance"] / "historical_scaffold_validation_README.txt", "QSAR-12", description="Historical validation-only scope statement")

PORTABLE_SCAFFOLD_DIR = DIR["Models"] / "Portable_Scaffold_Bundle"
scaffold_metadata, scaffold_arrays, scaffold_metadata_path, scaffold_arrays_path = serialize_preprocessor(scaffold_preprocessor, PORTABLE_SCAFFOLD_DIR, "scaffold")
scaffold_booster_path = PORTABLE_SCAFFOLD_DIR / "scaffold_xgboost.json"
SCAFFOLD_PIPELINE.named_steps["clf"].get_booster().save_model(str(scaffold_booster_path))
scaffold_calibration_path = PORTABLE_SCAFFOLD_DIR / "scaffold_external_sigmoid_calibration.json"
scaffold_calibration_path.write_text(json.dumps({
    "coef": float(SCAFFOLD_CALIBRATOR.coef_.ravel()[0]),
    "intercept": float(SCAFFOLD_CALIBRATOR.intercept_.ravel()[0]),
    "input": "logit(raw XGBoost probability)",
    "formula": "sigmoid(coef*logit(raw_probability)+intercept)",
}, indent=2), encoding="utf-8")
for generated_path, description in [
    (scaffold_metadata_path, "Portable scaffold preprocessor metadata"),
    (scaffold_arrays_path, "Portable scaffold scaler arrays"),
    (scaffold_booster_path, "Portable scaffold XGBoost model"),
    (scaffold_calibration_path, "Portable scaffold external calibration"),
]: register_export(generated_path, "QSAR-12", "portable_checkpoint", description)

portable_Z_test = portable_transform(X_FROZEN.iloc[test_rows], scaffold_metadata, scaffold_arrays)
portable_booster = xgb.Booster(); portable_booster.load_model(str(scaffold_booster_path))
portable_raw = portable_booster.predict(xgb.DMatrix(portable_Z_test))
coef = float(SCAFFOLD_CALIBRATOR.coef_.ravel()[0]); intercept = float(SCAFFOLD_CALIBRATOR.intercept_.ravel()[0])
portable_calibrated = 1.0 / (1.0 + np.exp(-(coef * safe_logit(portable_raw).ravel() + intercept)))
PORTABLE_SCAFFOLD_MAX_DIFF = float(np.max(np.abs(portable_calibrated - calibrated_scaffold_probability)))
PORTABLE_SCAFFOLD_TOLERANCE = 1e-12
assert PORTABLE_SCAFFOLD_MAX_DIFF <= PORTABLE_SCAFFOLD_TOLERANCE
write_json({
    "source_model_sha256": sha256_file(INPUTS["scaffold_pipeline"]),
    "source_calibrator_sha256": sha256_file(INPUTS["scaffold_calibrator"]),
    "n_features": len(scaffold_metadata["feature_names"]),
    "portable_vs_joblib_max_abs_probability_difference": PORTABLE_SCAFFOLD_MAX_DIFF,
    "numerical_equivalence_tolerance": PORTABLE_SCAFFOLD_TOLERANCE,
    "numerically_equivalent_within_tolerance": True,
    "purpose": "Numerically equivalent future replay of the validation-only corrected scaffold result.",
}, PORTABLE_SCAFFOLD_DIR / "portable_scaffold_bundle_manifest.json", "QSAR-12", "portable_checkpoint", "Portable scaffold checkpoint manifest")

plot_roc_pr_calibration(Y_SCAFFOLD_TEST, calibrated_scaffold_probability, "04_scaffold_heldout", "Corrected scaffold-disjoint", "QSAR-12")
plot_confusion(Y_SCAFFOLD_TEST, calibrated_scaffold_probability, 0.50, "Corrected scaffold confusion matrix", DIR["Figures"] / "04_scaffold_heldout_confusion_matrix.png", "QSAR-12")
fig, ax = plt.subplots(figsize=(6.5, 4.8))
ax.scatter(SCAFFOLD_LEVERAGE[~SCAFFOLD_INSIDE_AD], calibrated_scaffold_probability[~SCAFFOLD_INSIDE_AD], s=18, alpha=0.6, marker="^")
ax.scatter(SCAFFOLD_LEVERAGE[SCAFFOLD_INSIDE_AD], calibrated_scaffold_probability[SCAFFOLD_INSIDE_AD], s=18, alpha=0.45)
ax.axvline(SCAFFOLD_H_STAR, linestyle="--", linewidth=2, label=f"h* = {SCAFFOLD_H_STAR:.4f}")
ax.axhline(0.50, linestyle=":", linewidth=1.5)
ax.set(xlabel="Leverage against scaffold-training descriptor space", ylabel="Calibrated probability", title="Scaffold-held-out applicability domain")
ax.legend(); ax.grid(alpha=0.2); fig.tight_layout()
save_figure(fig, DIR["Applicability_Domain"] / "scaffold_qsar_leverage_probability_plot.png", "QSAR-12", "Scaffold-held-out leverage and calibrated probability")
log_event("QSAR-12", "Replayed corrected scaffold validation and verified portable numerical equivalence", {"metrics": SCAFFOLD_METRICS, "portable_max_diff": PORTABLE_SCAFFOLD_MAX_DIFF, "tolerance": PORTABLE_SCAFFOLD_TOLERANCE})


[2026-07-15T12:54:08.401562+00:00] QSAR-12: Replayed corrected scaffold validation and verified portable numerical equivalence


In [17]:
# QSAR-13 — Optional fresh scaffold retraining diagnostic (never authoritative)
FRESH_SCAFFOLD_DIAGNOSTIC = {"executed": False, "authoritative": False}
if RUN_FRESH_SCAFFOLD_RETRAIN_DIAGNOSTIC:
    calibration_rows = split_rows["calibration"]
    scale_ratio = float((Y[train_rows] == 0).sum() / (Y[train_rows] == 1).sum())
    fresh_pipeline = Pipeline([
        ("prep", DescriptorPreprocessor()),
        ("clf", xgb.XGBClassifier(
            n_estimators=500, max_depth=5, learning_rate=0.03,
            subsample=0.9, colsample_bytree=0.9,
            scale_pos_weight=scale_ratio, eval_metric="logloss",
            random_state=SEED, verbosity=0, tree_method="hist", n_jobs=N_JOBS,
        )),
    ])
    fresh_pipeline.fit(X_FROZEN.iloc[train_rows], Y[train_rows])
    raw_calibration = fresh_pipeline.predict_proba(X_FROZEN.iloc[calibration_rows])[:, 1]
    raw_test = fresh_pipeline.predict_proba(X_FROZEN.iloc[test_rows])[:, 1]
    fresh_calibrator = LogisticRegression(solver="lbfgs", random_state=SEED).fit(safe_logit(raw_calibration), Y[calibration_rows])
    fresh_probability = fresh_calibrator.predict_proba(safe_logit(raw_test))[:, 1]
    fresh_metrics = metric_row(Y_SCAFFOLD_TEST, fresh_probability, 0.50, "Fresh current-environment scaffold retraining diagnostic")
    FRESH_SCAFFOLD_DIAGNOSTIC = {
        "executed": True,
        "authoritative": False,
        "metrics": fresh_metrics,
        "reason_not_authoritative": "The exact historical XGBoost training version was not recorded; fresh retraining is known to differ by one classification from the archived checkpoint.",
    }
    write_csv(pd.DataFrame([fresh_metrics]), DIR["Validation"] / "Diagnostics" / "fresh_scaffold_retraining_metrics.csv", "QSAR-13", "diagnostic", "Non-authoritative current-environment scaffold retraining")
write_json(FRESH_SCAFFOLD_DIAGNOSTIC, DIR["Validation"] / "Diagnostics" / "fresh_scaffold_retraining_status.json", "QSAR-13", "diagnostic", "Fresh scaffold retraining status")
log_event("QSAR-13", "Processed optional fresh scaffold diagnostic", FRESH_SCAFFOLD_DIAGNOSTIC)


[2026-07-15T12:54:08.413413+00:00] QSAR-13: Processed optional fresh scaffold diagnostic


In [18]:
# QSAR-14 — Generate manuscript/supplementary outputs and claim-to-cell mapping
xgb_cv = CV_FOR_REPORT.loc[CV_FOR_REPORT["model"].eq("XGBoost")].iloc[0].to_dict()
rf_cv = CV_FOR_REPORT.loc[CV_FOR_REPORT["model"].eq("Random_Forest")].iloc[0].to_dict()
svm_cv = CV_FOR_REPORT.loc[CV_FOR_REPORT["model"].eq("SVM_RBF")].iloc[0].to_dict()

manuscript_rows = [
    {"evidence_layer": "AURKB bioactivity curation", "result": "3,248 IC50 records reduced to 1,854 unique compounds; 1,429 active, 425 inactive; median pIC50 6.9281; 862 scaffolds", "status": "REGENERATED/VERIFIED", "source_section": "QSAR-04"},
    {"evidence_layer": "Descriptor-QSAR model selection", "result": f"217 descriptors; 165 retained; XGBoost CV ROC-AUC {xgb_cv['roc_auc_mean']:.4f} ± {xgb_cv['roc_auc_std']:.4f}; PR-AUC {xgb_cv['average_precision_mean']:.4f} ± {xgb_cv['average_precision_std']:.4f}; BA {xgb_cv['balanced_accuracy_mean']:.4f} ± {xgb_cv['balanced_accuracy_std']:.4f}; MCC {xgb_cv['mcc_mean']:.4f} ± {xgb_cv['mcc_std']:.4f}", "status": "HASH-VERIFIED ARCHIVED AUTHORITATIVE; FRESH RETRAINING COMPARISON EXPORTED", "source_section": "QSAR-05/QSAR-07"},
    {"evidence_layer": "QSAR random held-out validation", "result": f"n=371; ROC-AUC {RANDOM_METRICS_FOR_REPORT['roc_auc']:.4f}; PR-AUC {RANDOM_METRICS_FOR_REPORT['pr_auc']:.4f}; BA {RANDOM_METRICS_FOR_REPORT['balanced_accuracy']:.4f}; MCC {RANDOM_METRICS_FOR_REPORT['mcc']:.4f}; sensitivity {RANDOM_METRICS_FOR_REPORT['sensitivity']:.4f}; specificity {RANDOM_METRICS_FOR_REPORT['specificity']:.4f}; Brier {RANDOM_METRICS_FOR_REPORT['brier_score']:.4f}", "status": "FORENSIC-AUDIT VERIFIED HISTORICAL AGGREGATE; FRESH RETRAINING COMPARISON EXPORTED", "source_section": "QSAR-08"},
    {"evidence_layer": "Scaffold-disjoint QSAR validation", "result": f"Train/calibration/test 1,062/348/444; ROC-AUC {SCAFFOLD_METRICS['roc_auc']:.4f}; PR-AUC {SCAFFOLD_METRICS['pr_auc']:.4f}; BA {SCAFFOLD_METRICS['balanced_accuracy']:.4f}; MCC {SCAFFOLD_METRICS['mcc']:.4f}; sensitivity {SCAFFOLD_METRICS['sensitivity']:.4f}; specificity {SCAFFOLD_METRICS['specificity']:.4f}; Brier {SCAFFOLD_METRICS['brier_score']:.4f}", "status": "EXACT CHECKPOINT REPLAY", "source_section": "QSAR-12"},
    {"evidence_layer": "QSAR robustness", "result": f"Observed CV ROC-AUC {Y_RANDOMIZATION_SUMMARY['observed_cv_roc_auc']:.4f} vs null {Y_RANDOMIZATION_SUMMARY['permuted_mean_roc_auc']:.4f} ± {Y_RANDOMIZATION_SUMMARY['permuted_std_roc_auc']:.4f}; p={Y_RANDOMIZATION_SUMMARY['permutation_pvalue']:.4f}", "status": "HASH-VERIFIED 30-SCORE VECTOR; SUMMARY RECOMPUTED", "source_section": "QSAR-09"},
    {"evidence_layer": "Production screening transfer", "result": "86,056 input rows verified; reported 34,721 inside AD and 31,020 predicted active are not exactly regenerable; exact 6,232-row downstream transfer snapshot preserved", "status": "PARTIAL — UPSTREAM IRREPRODUCIBLE", "source_section": "QSAR-11"},
]
MANUSCRIPT_SUMMARY = pd.DataFrame(manuscript_rows)
write_csv(MANUSCRIPT_SUMMARY, DIR["Manuscript_Output"] / "Table_QSAR_reproduction_summary.csv", "QSAR-14", "publication_output", "Manuscript-ready QSAR summary with status labels")

# Copy or write core supplementary outputs into a clearly named package.
write_csv(CURATION_LOG_REPLAY, DIR["Supplementary_Output"] / "S_QSAR_curation_steps_replayed.csv", "QSAR-14", "publication_output", "Supplementary curation steps")
write_csv(CV_FOR_REPORT, DIR["Supplementary_Output"] / "S_QSAR_model_comparison_cross_validation.csv", "QSAR-14", "publication_output", "Supplementary model comparison")
write_csv(pd.DataFrame([RANDOM_METRICS_FOR_REPORT]), DIR["Supplementary_Output"] / "S_QSAR_random_heldout_metrics_authoritative.csv", "QSAR-14", "publication_output_historical_verified", "Supplementary authoritative random held-out aggregate metrics")
if RANDOM_EXACT_REPRODUCTION and RANDOM_PREDICTIONS is not None:
    write_csv(RANDOM_PREDICTIONS, DIR["Supplementary_Output"] / "S_QSAR_random_heldout_predictions_exact_current_environment.csv", "QSAR-14", "publication_output_exact_reproduction", "Supplementary per-compound probabilities only when fresh fitting is exact")
write_csv(pd.DataFrame([SCAFFOLD_METRICS, SCAFFOLD_INSIDE_METRICS]), DIR["Supplementary_Output"] / "S_QSAR_scaffold_metrics.csv", "QSAR-14", "publication_output", "Supplementary corrected scaffold metrics")
write_csv(scaffold_predictions, DIR["Supplementary_Output"] / "S_QSAR_scaffold_predictions.csv", "QSAR-14", "publication_output", "Supplementary corrected scaffold predictions")
write_csv(pd.DataFrame({"permutation": np.arange(1, len(permutation_scores) + 1), "permuted_roc_auc": permutation_scores}), DIR["Supplementary_Output"] / "S_QSAR_y_randomization_scores_authoritative.csv", "QSAR-14", "publication_output_historical_verified", "Supplementary hash-verified historical Y-randomization scores")
copy_source(INPUTS["candidate_snapshot"], DIR["Supplementary_Output"] / "S_QSAR_exact_6232_transfer_snapshot.csv", "QSAR-14", "publication_output_archived", "Supplementary exact historical QSAR-to-GCN transfer snapshot")

MAPPING_ROWS = [
    ["Manuscript Methods P0013; Results P0055; Table 1", "Supplementary curation tables", "Raw ChEMBL IC50 records", "3,248", "VERIFIED", "QSAR-04", "CSV/CHEMBL2185_IC50_ChEMBL_raw_authoritative.csv", "Frozen raw table hash-verified; count recomputed"],
    ["Manuscript Methods P0013; Results P0055", "Supplementary curation log", "Curated compounds", "1,854", "VERIFIED", "QSAR-04", "Validation/curation_replay_current_environment.csv", "Curation replay preserves molecular identities and labels"],
    ["Manuscript Results P0055; Table 1", "Curated training-set files", "Active/inactive classes", "1,429 / 425", "VERIFIED", "QSAR-04", "Tables/QSAR_input_audit.json", "Computed from frozen curated table"],
    ["Manuscript Results P0055; Table 1", "Curated training-set files", "Median pIC50 and scaffolds", "6.9281179927; 862", "VERIFIED", "QSAR-04", "Tables/QSAR_input_audit.json", "Direct executable recomputation"],
    ["Manuscript Methods P0016; Table 1", "Supplementary descriptor table", "Raw RDKit descriptors", "1,854 × 217", "VERIFIED FROZEN MATRIX", "QSAR-05", "CSV/curated_training_rdkit_descriptors_raw_authoritative.csv", "Current RDKit drift is separately reported"],
    ["Manuscript Results P0066; Table 1", "Supplementary model comparison", "Selected descriptors", "165", "VERIFIED", "QSAR-05", "Tables/selected_rdkit_descriptors_regenerated.csv", "Reproduced from frozen matrix"],
    ["Manuscript Methods P0016", "Supplementary methods", "Preprocessing", "20% missingness; median imputation; zero variance; |r|>0.95; z-score", "VERIFIED", "QSAR-03/QSAR-05", "Tables/descriptor_preprocessing_manifest.json", "Fitted only within training folds/subsets"],
    ["Manuscript Methods P0016; Results P0066", "Supplementary model comparison", "Random split", "1,483 train / 371 test; seed 42", "VERIFIED", "QSAR-06", "CSV/random_seed42_split_assignment.csv", "Exact stratified assignment exported"],
    ["Manuscript Results P0066; Table 1", "Supplementary Table S4", "XGBoost five-fold CV metrics", "ROC 0.8886108557; PR 0.9596540533; BA 0.7824009356; MCC 0.5814270866", "VERIFIED", "QSAR-07", "Tables/model_comparison_cross_validation_authoritative.csv", "Hash-verified archived source; fresh environment comparison exported"],
    ["Manuscript Results P0066", "Supplementary Table S4", "SVM-RBF five-fold CV metrics", "ROC 0.8637292081; PR 0.9524232475; BA 0.7844679609; MCC 0.5295461976", "VERIFIED", "QSAR-07", "Tables/model_comparison_cross_validation_authoritative.csv", "Hash-verified archived source; fresh environment comparison exported"],
    ["Manuscript Results P0066", "Supplementary Table S4", "Random-forest five-fold CV metrics", "ROC 0.8846090148; PR 0.9557837236; BA 0.7237216146; MCC 0.5326274206", "PARTIALLY VERIFIED", "QSAR-07", "Validation/model_comparison_archived_vs_regenerated.csv", "Version-sensitive scikit-learn result"],
    ["Manuscript Results P0066; Table 1", "Supplementary random held-out table/figures", "Calibrated random held-out metrics", "ROC 0.9062937063; PR 0.9665234428; BA 0.7903126285; MCC 0.6106527518; Brier 0.0915081198", "VERIFIED", "QSAR-08", "Tables/random_heldout_validation_metrics_authoritative.csv", "Forensic-audit verified historical aggregate; fresh comparison exported"],
    ["Manuscript Results P0066", "Supplementary random held-out table", "Random held-out confusion and threshold metrics", "TN55 FP30 FN19 TP267; sensitivity 0.9335664336; specificity 0.6470588235; precision 0.8989898990; F1 0.9159519726", "VERIFIED AGGREGATE; PER-COMPOUND HISTORICAL VECTOR ABSENT", "QSAR-08", "Validation/Diagnostics/random_heldout_predictions_current_environment.csv", "Aggregate values were independently reproduced; fresh per-compound probabilities are diagnostic unless all aggregate metrics match exactly"],
    ["Manuscript Methods P0018; Results P0068; Table 1", "Supplementary Y-randomization", "Y-randomization", "Observed 0.8886108557; null 0.5048808074 ± 0.0238988150; p=0.0322580645; n=30", "VERIFIED", "QSAR-09", "Y_Randomization/y_randomization_summary_authoritative.json", "Summary recomputed from hash-verified historical 30-score vector"],
    ["Manuscript Methods P0018", "Model deposit", "Production calibrated model and AD artifacts", "3 original artifacts", "VERIFIED", "QSAR-10", "Models/Historical_Archived; Applicability_Domain", "Original checkpoints copied and portable bundle generated"],
    ["Manuscript Methods P0018; Supplementary screening summary", "Supplementary screening table", "Production leverage threshold", "h*=0.2686084142394822", "VERIFIED", "QSAR-10", "Applicability_Domain/production_applicability_domain_manifest.json", "Exact from 1,854 × 165 reference matrix"],
    ["Manuscript Results P0068", "Supplementary screening summary", "Full-library valid/invalid rows", "86,056 / 0", "VERIFIED COVERAGE", "QSAR-11", "Validation/full_library_screening_reproducibility_status.json", "Input rows exact; original probability vector absent"],
    ["Manuscript Results P0068; Table 1", "Supplementary screening summary", "Inside descriptor AD", "34,721", "NOT VERIFIED", "QSAR-11", "Validation/full_library_screening_reproducibility_status.json", "Original all-screened CSV and exact environment missing"],
    ["Manuscript Results P0068", "Supplementary screening summary", "Predicted active at probability ≥0.50", "31,020", "NOT VERIFIED", "QSAR-11", "Validation/full_library_screening_reproducibility_status.json", "Current replay 31,018; version-sensitive inference"],
    ["Manuscript Results P0068; Table 2 input", "Supplementary screening summary", "High-confidence inside-AD at probability ≥0.75", "6,232", "NOT VERIFIED UPSTREAM; SNAPSHOT PRESERVED", "QSAR-11", "CSV/Archived_Historical_Screening/AURKB_QSAR_to_GCN_exact_6232_candidate_snapshot.csv", "Exact downstream snapshot preserved; upstream regeneration unavailable"],
    ["Manuscript Methods P0017; Results P0067; Table 1", "Supplementary scaffold split table", "Scaffold train/calibration/test", "1,062 / 348 / 444; 516 / 173 / 173 scaffolds", "VERIFIED", "QSAR-12", "Tables/scaffold_split_summary_regenerated.csv", "Exact assignment"],
    ["Manuscript Methods P0017", "Supplementary scaffold overlap audit", "Scaffold overlap", "0 / 0 / 0", "VERIFIED", "QSAR-12", "Validation/scaffold_split_overlap_regenerated.json", "Set intersections recomputed"],
    ["Manuscript Methods P0017", "Supplementary scaffold AD", "Train-only selected features and h*", "161; 0.4576271186440678", "VERIFIED", "QSAR-12", "Models/Portable_Scaffold_Bundle; Applicability_Domain", "Exact checkpoint replay"],
    ["Manuscript Results P0067; Table 1", "Supplementary scaffold metrics", "Scaffold-test performance", "ROC 0.8530772294; PR 0.9541811835; BA 0.6797628894; MCC 0.4279409347; Brier 0.1160417224", "VERIFIED", "QSAR-12", "Tables/scaffold_qsar_metrics_regenerated.csv", "Executable checkpoint inference"],
    ["Manuscript Results P0067", "Supplementary scaffold metrics", "Scaffold-test confusion", "TN39 FP54 FN21 TP330; sensitivity 0.9401709402; specificity 0.4193548387", "VERIFIED", "QSAR-12", "CSV/scaffold_qsar_predictions_regenerated.csv", "Computed from replayed probabilities"],
    ["Manuscript Results P0067", "Supplementary inside-AD scaffold metrics", "Inside-AD scaffold test", "n420; ROC 0.8551941610; PR 0.9568648337; BA 0.6845238095; MCC 0.4342571564; Brier 0.1120513825", "VERIFIED", "QSAR-12", "Tables/scaffold_qsar_metrics_regenerated.csv", "Leverage and metrics recomputed"],
    ["Manuscript Figure 2 A-C", "Supplementary QSAR CV/random validation figures", "Random-split figures", "CV, ROC, PR, calibration, confusion", "EXACT ONLY WHEN FRESH ENVIRONMENT MATCHES; OTHERWISE DIAGNOSTIC", "QSAR-07/QSAR-08", "Validation/Diagnostics; Figures; Calibration", "Current-environment figures are not presented as exact when retraining drifts"],
    ["Manuscript Figure 2 D-F", "Supplementary scaffold figures", "Corrected scaffold figures", "ROC, PR, calibration, confusion, leverage-probability", "VERIFIED/REGENERATED", "QSAR-12", "Figures; Calibration; Applicability_Domain", "Generated from exact checkpoint replay"],
]
MAPPING_COLUMNS = ["manuscript_location", "supplementary_location", "result", "reported_value", "status", "notebook_section", "primary_output", "evidence_note"]
RESULT_MAPPING = pd.DataFrame(MAPPING_ROWS, columns=MAPPING_COLUMNS)
write_csv(RESULT_MAPPING, DIR["Provenance"] / "QSAR_Result_to_Notebook_Section_Mapping.csv", "QSAR-14", "provenance", "Claim-to-notebook and output mapping")
write_csv(RESULT_MAPPING, DIR["Manuscript_Output"] / "QSAR_Result_to_Notebook_Section_Mapping.csv", "QSAR-14", "publication_output", "Publication-facing result traceability table")

IRREPRODUCIBLE = pd.DataFrame([
    {"result": "Full-library inside-AD count", "reported_value": "34,721", "status": "NOT VERIFIED", "precise_reason": "The original 86,056-row all-screened probability/leverage CSV is absent. Descriptor generation changes with RDKit version, and the exact QSAR runtime was not fully recorded.", "preserved_evidence": "Prescreen input, production checkpoint, AD artifacts, forensic replay report"},
    {"result": "Full-library predicted active count at 0.50", "reported_value": "31,020", "status": "NOT VERIFIED", "precise_reason": "The original full probability vector is absent; current checkpoint/environment replay produced 31,018.", "preserved_evidence": "Production checkpoint and forensic replay report"},
    {"result": "Upstream regeneration of 6,232 high-confidence inside-AD candidates", "reported_value": "6,232", "status": "NOT VERIFIED UPSTREAM", "precise_reason": "The exact 6,232-row snapshot is present and intact, but the missing all-screened file and unpinned descriptor/training environment prevent exact re-selection from 86,056 compounds.", "preserved_evidence": "Exact gcn_candidate_input_audit.csv snapshot"},
    {"result": "Random-forest five-fold CV row", "reported_value": "ROC 0.8846090148", "status": "PARTIALLY VERIFIED", "precise_reason": "The archived row is internally consistent; scikit-learn implementation/version drift changes the regenerated RF result.", "preserved_evidence": "Archived model comparison table and regenerated comparison"},
    {"result": "Exact descriptor regeneration from SMILES in the validated current environment", "reported_value": "217-column historical matrix", "status": "PARTIALLY VERIFIED", "precise_reason": "Current RDKit changes NumHAcceptors values. The hash-verified historical descriptor matrix is therefore the authoritative input.", "preserved_evidence": "Frozen matrix and descriptor drift report"},
    {"result": "Exact fresh retraining of corrected scaffold model", "reported_value": "Archived scaffold metrics", "status": "PARTIALLY VERIFIED", "precise_reason": "The original XGBoost training version was not recorded. The archived train-only checkpoint and calibrator replay exactly, while fresh retraining is version-sensitive.", "preserved_evidence": "Scaffold model/calibrator and portable exact bundle"},
    {"result": "Bitwise-exact fresh random-branch retraining on every platform", "reported_value": "Archived CV, random held-out, and Y-randomization values", "status": "ENVIRONMENT-SENSITIVE", "precise_reason": "XGBoost/scikit-learn numerical behavior can differ across operating systems, compiled libraries, and package builds. The notebook now exports fresh comparisons without replacing the hash-verified historical layer.", "preserved_evidence": "Archived CV table, archived Y-randomization vector, forensic-audit verified held-out metrics, current-environment comparison files"},
])
write_csv(IRREPRODUCIBLE, DIR["Provenance"] / "Remaining_Irreproducible_QSAR_Results.csv", "QSAR-14", "provenance", "Remaining reproducibility gaps and precise causes")
write_csv(IRREPRODUCIBLE, DIR["Manuscript_Output"] / "Remaining_Irreproducible_QSAR_Results.csv", "QSAR-14", "publication_output", "Publication-facing disclosure of remaining gaps")

# Composite figure is generated only when the full random branch has produced all required panels.
required_panels = [
    DIR["Figures"] / "02_cv_roc_auc.png",
    DIR["Figures"] / "02_cv_mcc.png",
    DIR["Figures"] / "03_random_heldout_roc_curve.png",
    DIR["Figures"] / "03_random_heldout_precision_recall_curve.png",
    DIR["Figures"] / "04_scaffold_heldout_roc_curve.png",
    DIR["Figures"] / "04_scaffold_heldout_precision_recall_curve.png",
]
if RANDOM_EXACT_REPRODUCTION and CV_EXACT_REPRODUCTION and all(path.exists() for path in required_panels):
    fig, axes = plt.subplots(3, 2, figsize=(12, 15))
    labels = ["A", "B", "C", "D", "E", "F"]
    for ax, path, label in zip(axes.ravel(), required_panels, labels):
        ax.imshow(plt.imread(path)); ax.axis("off"); ax.set_title(label, loc="left", fontweight="bold")
    fig.tight_layout()
    save_figure(fig, DIR["Manuscript_Output"] / "Figure_2_QSAR_Validation_Reproduced.png", "QSAR-14", "Composite manuscript QSAR validation figure")
else:
    write_json({"status": "not_generated_as_authoritative_because_full_random_branch_was_not_exact_or_not_executed", "required_panels": [str(p.relative_to(RUN_DIR)) for p in required_panels]}, DIR["Manuscript_Output"] / "Figure_2_QSAR_Validation_Reproduced_status.json", "QSAR-14", "audit_documentation", "Composite figure generation status")
log_event("QSAR-14", "Generated manuscript/supplementary outputs and result mapping", {"mapping_rows": len(RESULT_MAPPING), "irreproducible_rows": len(IRREPRODUCIBLE)})


[2026-07-15T12:54:08.473290+00:00] QSAR-14: Generated manuscript/supplementary outputs and result mapping


In [19]:
# QSAR-15 — Final environment capture, execution log, SHA256 inventory, and no-overwrite verification
PACKAGE_NAMES = ["numpy", "pandas", "scikit-learn", "xgboost", "rdkit", "joblib", "matplotlib", "jupyter", "nbformat"]
PACKAGE_VERSIONS = {}
for package_name in PACKAGE_NAMES:
    try:
        PACKAGE_VERSIONS[package_name] = importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        PACKAGE_VERSIONS[package_name] = "not installed / metadata unavailable"
PACKAGE_VERSIONS.update({
    "python": platform.python_version(),
    "rdkit_runtime": rdBase.rdkitVersion,
    "scikit_learn_runtime": sklearn.__version__,
    "xgboost_runtime": xgb.__version__,
    "numpy_runtime": np.__version__,
    "pandas_runtime": pd.__version__,
})
write_json(PACKAGE_VERSIONS, DIR["Provenance"] / "package_versions.json", "QSAR-15", "provenance", "Runtime package versions")

ENVIRONMENT = {
    "python_executable": sys.executable,
    "python_version": sys.version,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "project_root": str(BASE_DIR),
    "process_working_directory": str(Path.cwd().resolve()),
    "input_archive_directory": str(INPUT_ARCHIVE_DIR),
    "environment_variables": {
        "QSAR_EXECUTION_PROFILE": EXECUTION_PROFILE,
        "QSAR_N_JOBS": N_JOBS,
        "QSAR_PROJECT_ROOT": os.environ.get("QSAR_PROJECT_ROOT", "not set; configuration cell or auto-detection used"),
        "PYTHONHASHSEED": os.environ.get("PYTHONHASHSEED", "not set before interpreter start"),
    },
    "historical_environment_evidence": {
        "sklearn_from_joblib_serialization": "1.6.1",
        "same_day_prescreen_python": "3.13.5",
        "same_day_prescreen_numpy": "2.2.5",
        "same_day_prescreen_pandas": "2.2.3",
        "same_day_prescreen_rdkit": "2026.03.2",
        "historical_qsar_xgboost": "not directly recoverable",
        "note": "Same-day prescreen versions are supporting evidence, not a direct QSAR environment record.",
    },
}
write_json(ENVIRONMENT, DIR["Provenance"] / "environment_information.json", "QSAR-15", "provenance", "Runtime and historical environment evidence")
write_json({
    "python_random_seed": SEED,
    "numpy_seed": SEED,
    "split_seed": SEED,
    "cross_validation_seed": SEED,
    "model_random_state": SEED,
    "permutation_seed": SEED,
}, DIR["Provenance"] / "random_seeds.json", "QSAR-15", "provenance", "All deterministic random seeds")

RUN_ENDED_UTC = datetime.now(timezone.utc)
TIMESTAMP_REPORT = {
    "run_id": RUN_ID,
    "started_utc": RUN_STARTED_UTC.isoformat(),
    "ended_utc": RUN_ENDED_UTC.isoformat(),
    "elapsed_seconds": (RUN_ENDED_UTC - RUN_STARTED_UTC).total_seconds(),
    "execution_profile": EXECUTION_PROFILE,
}
write_json(TIMESTAMP_REPORT, DIR["Provenance"] / "timestamp_report.json", "QSAR-15", "provenance", "Execution timestamps")

# Confirm original archives were unchanged.
SOURCE_ARCHIVE_POSTCHECK = []
for path_string, before in SOURCE_ARCHIVE_STATS.items():
    path = Path(path_string)
    after = {"size_bytes": path.stat().st_size, "mtime_ns": path.stat().st_mtime_ns, "sha256": sha256_file(path)}
    unchanged = before == after
    SOURCE_ARCHIVE_POSTCHECK.append({"path": path_string, "unchanged": unchanged, "before_sha256": before["sha256"], "after_sha256": after["sha256"], "before_size": before["size_bytes"], "after_size": after["size_bytes"]})
    if not unchanged:
        raise RuntimeError(f"Source archive changed during execution: {path}")
write_csv(pd.DataFrame(SOURCE_ARCHIVE_POSTCHECK), DIR["Provenance"] / "source_archive_no_overwrite_verification.csv", "QSAR-15", "provenance", "Proof that original source archives were unchanged")

# Register execution log before creating the final inventory.
log_event("QSAR-15", "Finalizing manifests and output hashes")
execution_log_path = register_export(DIR["Logs"] / "execution_log.jsonl", "QSAR-15", "provenance", "Timestamped execution event log")
with execution_log_path.open("w", encoding="utf-8") as handle:
    for event in EXECUTION_EVENTS:
        handle.write(json.dumps(event, ensure_ascii=False, default=str) + "\n")
write_csv(pd.DataFrame(EXECUTION_EVENTS), DIR["Logs"] / "execution_log.csv", "QSAR-15", "provenance", "Tabular execution event log")

# Build traceable file inventory. The registry ties each output to the section that created it.
registry_by_path = {row["relative_path"]: row for row in EXPORT_REGISTRY}
file_inventory = []
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        relative = str(path.relative_to(RUN_DIR))
        registered = registry_by_path.get(relative, {})
        file_inventory.append({
            "relative_path": relative,
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
            "source_section": registered.get("source_section", "QSAR-15/finalization"),
            "provenance_class": registered.get("provenance_class", "generated_unregistered_intermediate"),
            "description": registered.get("description", "Generated during notebook execution"),
        })
FILE_INVENTORY = pd.DataFrame(file_inventory)
write_csv(FILE_INVENTORY, DIR["Provenance"] / "exported_file_inventory_sha256.csv", "QSAR-15", "provenance", "Complete exported-file SHA256 inventory")

FINAL_MANIFEST = {
    "architecture": "Option A: one authoritative notebook with isolated production and corrected-scaffold branches",
    "run_id": RUN_ID,
    "run_directory": str(RUN_DIR),
    "execution_profile": EXECUTION_PROFILE,
    "source_archives_unchanged": bool(pd.DataFrame(SOURCE_ARCHIVE_POSTCHECK)["unchanged"].all()),
    "all_outputs_inside_isolated_directory": all(RUN_DIR.resolve() in path.resolve().parents for path in RUN_DIR.rglob("*") if path.is_file()),
    "critical_input_hashes_all_matched": bool(pd.DataFrame(HASH_CHECKS)["matched"].all()),
    "exact_results": [
        "Curation counts, labels, activities and scaffolds",
        "Frozen descriptor matrix and 165-feature selection",
        "XGBoost and SVM five-fold CV in full profile",
        "Random calibrated held-out validation in full profile",
        "Y-randomization in full profile",
        "Production AD h-star and archived checkpoint inference",
        "Corrected scaffold split, probabilities, metrics, AD and figures",
        "Exact 6,232-row QSAR-to-GCN transfer snapshot integrity",
    ],
    "remaining_irreproducible": IRREPRODUCIBLE.to_dict(orient="records"),
    "portable_production_checkpoint_max_abs_difference": PORTABLE_PRODUCTION_MAX_DIFF,
    "portable_scaffold_checkpoint_max_abs_difference": PORTABLE_SCAFFOLD_MAX_DIFF,
    "file_inventory_path": "Provenance/exported_file_inventory_sha256.csv",
    "result_mapping_path": "Provenance/QSAR_Result_to_Notebook_Section_Mapping.csv",
}
write_json(FINAL_MANIFEST, DIR["Provenance"] / "provenance_manifest.json", "QSAR-15", "provenance", "Final authoritative provenance manifest")

# Refresh inventory once to include the manifest and inventory files themselves.
file_inventory_final = []
registry_by_path = {row["relative_path"]: row for row in EXPORT_REGISTRY}
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        relative = str(path.relative_to(RUN_DIR))
        registered = registry_by_path.get(relative, {})
        file_inventory_final.append({
            "relative_path": relative,
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
            "source_section": registered.get("source_section", "QSAR-15/finalization"),
            "provenance_class": registered.get("provenance_class", "generated_finalization"),
            "description": registered.get("description", "Generated during finalization"),
        })
pd.DataFrame(file_inventory_final).to_csv(DIR["Provenance"] / "exported_file_inventory_sha256.csv", index=False)

print("\nFINAL AUTHORITATIVE QSAR RUN COMPLETE")
print("Run directory:", RUN_DIR)
print("Execution profile:", EXECUTION_PROFILE)
print("Original archives unchanged:", all(row["unchanged"] for row in SOURCE_ARCHIVE_POSTCHECK))
print("All generated files are isolated under QSAR_Final_Reproduction:", FINAL_MANIFEST["all_outputs_inside_isolated_directory"])
print("Result mapping:", DIR["Provenance"] / "QSAR_Result_to_Notebook_Section_Mapping.csv")
print("Remaining gaps:", DIR["Provenance"] / "Remaining_Irreproducible_QSAR_Results.csv")


[2026-07-15T12:54:08.671318+00:00] QSAR-15: Finalizing manifests and output hashes

FINAL AUTHORITATIVE QSAR RUN COMPLETE
Run directory: C:\Users\Prottoy\Downloads\Notebook_2_Retrail\Inputs\QSAR_Final_Reproduction\run_20260715T124827Z_79cebaa2
Execution profile: full
Original archives unchanged: True
All generated files are isolated under QSAR_Final_Reproduction: True
Result mapping: C:\Users\Prottoy\Downloads\Notebook_2_Retrail\Inputs\QSAR_Final_Reproduction\run_20260715T124827Z_79cebaa2\Provenance\QSAR_Result_to_Notebook_Section_Mapping.csv
Remaining gaps: C:\Users\Prottoy\Downloads\Notebook_2_Retrail\Inputs\QSAR_Final_Reproduction\run_20260715T124827Z_79cebaa2\Provenance\Remaining_Irreproducible_QSAR_Results.csv


In [20]:
from pathlib import Path
import os
import pandas as pd

print("Current working directory:")
print(Path.cwd().resolve())

print("\nRUNDIR:")
print(globals().get("RUNDIR", "<not defined>"))

print("\nOUTPUTROOT:")
print(globals().get("OUTPUTROOT", "<not defined>"))

print("\nBASEDIR:")
print(globals().get("BASEDIR", "<not defined>"))

print("\nCandidate-related variables currently defined:")
for name in sorted(globals()):
    lower = name.lower()
    if any(token in lower for token in ["gcn", "candidate", "screen", "transfer"]):
        value = globals()[name]
        if isinstance(value, (str, Path)):
            print(f"{name} = {value}")

roots = []

for value in [
    globals().get("RUNDIR"),
    globals().get("OUTPUTROOT"),
    globals().get("BASEDIR"),
    globals().get("INPUTARCHIVEDIR"),
    globals().get("ARCHIVEDIR"),
    Path.cwd(),
    Path.cwd().parent,
]:
    if value is None:
        continue

    try:
        path = Path(value).expanduser().resolve()
        if path.exists() and path not in roots:
            roots.append(path)
    except Exception:
        pass

patterns = [
    "*.csv",
    "*.CSV",
]

all_csvs = []

for root in roots:
    try:
        for pattern in patterns:
            all_csvs.extend(root.rglob(pattern))
    except (PermissionError, OSError):
        pass

all_csvs = sorted(
    {path.resolve() for path in all_csvs if path.is_file()},
    key=lambda path: (path.stat().st_mtime, str(path)),
    reverse=True,
)

keywords = [
    "gcn",
    "candidate",
    "screen",
    "highconfidence",
    "insidead",
    "prescreen",
    "transfer",
    "qsar",
]

relevant_csvs = [
    path for path in all_csvs
    if any(keyword in path.name.lower() for keyword in keywords)
]

print(f"\nTotal CSV files found: {len(all_csvs)}")
print(f"Relevant CSV files found: {len(relevant_csvs)}\n")

for index, path in enumerate(relevant_csvs[:200], start=1):
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"{index:>3}. {path}  [{size_mb:.2f} MB]")

Current working directory:
C:\Users\Prottoy\Downloads\Notebook_2_Retrail

RUNDIR:
<not defined>

OUTPUTROOT:
<not defined>

BASEDIR:
<not defined>

Candidate-related variables currently defined:
transfer_path = C:\Users\Prottoy\Downloads\Notebook_2_Retrail\Inputs\QSAR_Final_Reproduction\run_20260715T124827Z_79cebaa2\CSV\Archived_Historical_Screening\AURKB_QSAR_to_GCN_exact_6232_candidate_snapshot.csv
transfer_sha256 = 8821947e92d8b209d291b627bae3b1d5068c6e2f1c0154f695abf0e73673f4df

Total CSV files found: 509
Relevant CSV files found: 157

  1. C:\Users\Prottoy\Downloads\Notebook_2_Retrail\Inputs\QSAR_Final_Reproduction\run_20260715T124827Z_79cebaa2\Manuscript_Output\Remaining_Irreproducible_QSAR_Results.csv  [0.00 MB]
  2. C:\Users\Prottoy\Downloads\Notebook_2_Retrail\Inputs\QSAR_Final_Reproduction\run_20260715T124827Z_79cebaa2\Provenance\Remaining_Irreproducible_QSAR_Results.csv  [0.00 MB]
  3. C:\Users\Prottoy\Downloads\Notebook_2_Retrail\Inputs\QSAR_Final_Reproduction\run_20260715T

In [21]:
from pathlib import Path
import hashlib
import pandas as pd

EXPECTED_SHA256 = (
    "8821947e92d8b209d291b627bae3b1d5068c6e2f1c0154f695abf0e73673f4df"
)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)

    return digest.hexdigest()

confirmed_path = Path(transfer_path)

if not confirmed_path.is_file():
    raise FileNotFoundError(
        f"Verified QSAR-to-GCN transfer file is missing:\n{confirmed_path}"
    )

actual_sha256 = sha256_file(confirmed_path)

df = pd.read_csv(confirmed_path, low_memory=False)

print("Verified QSAR-to-GCN transfer file:")
print(confirmed_path)

print("\nSHA-256:")
print(actual_sha256)

print("\nExpected SHA-256:")
print(EXPECTED_SHA256)

print("\nRows:", len(df))
print("Columns:", df.columns.tolist())

assert actual_sha256 == EXPECTED_SHA256, (
    "FAIL: transfer file hash does not match the archived GCN "
    "candidate-input snapshot."
)

assert len(df) == 6232, (
    f"FAIL: expected 6,232 candidate rows; found {len(df):,}."
)

required_columns = {
    "identifier",
    "smiles",
    "prob_active_qsar",
    "inside_AD",
}

missing_columns = required_columns - set(df.columns)

assert not missing_columns, (
    f"FAIL: missing required GCN-transfer columns: "
    f"{sorted(missing_columns)}"
)

print("\nPASS")
print(
    "The QSAR notebook produced an exact byte-identical copy of "
    "the archived 6,232-row GCN candidate-input snapshot."
)

Verified QSAR-to-GCN transfer file:
C:\Users\Prottoy\Downloads\Notebook_2_Retrail\Inputs\QSAR_Final_Reproduction\run_20260715T124827Z_79cebaa2\CSV\Archived_Historical_Screening\AURKB_QSAR_to_GCN_exact_6232_candidate_snapshot.csv

SHA-256:
8821947e92d8b209d291b627bae3b1d5068c6e2f1c0154f695abf0e73673f4df

Expected SHA-256:
8821947e92d8b209d291b627bae3b1d5068c6e2f1c0154f695abf0e73673f4df

Rows: 6232
Columns: ['identifier', 'smiles', 'prob_active_qsar', 'prediction_qsar', 'leverage', 'inside_AD', 'high_confidence_qsar_inside_AD']

PASS
The QSAR notebook produced an exact byte-identical copy of the archived 6,232-row GCN candidate-input snapshot.


## Authoritative interpretation

- The **random-split QSAR branch** must be executed in `full` profile for independent retraining of the exact XGBoost/SVM CV, calibrated held-out result, and 30-permutation Y-randomization.
- The **corrected scaffold branch** is reproduced by executable inference from the archived train-only checkpoint and external calibrator. Portable XGBoost/preprocessing/calibration bundles are generated and verified to be numerically equivalent to the original joblib checkpoints within a strict 1e-12 tolerance.
- The **historical 217-descriptor matrix** is the authoritative numerical input. Descriptor regeneration is retained as a version-drift diagnostic rather than silently replacing historical values.
- The exact **6,232-row QSAR-to-GCN transfer snapshot** is preserved without alteration. The upstream counts 34,721, 31,020, and exact re-selection of 6,232 cannot be regenerated because the original all-screened CSV and complete historical QSAR environment are absent.
- No Prescreening or GCN notebook is modified by this notebook.
